# Particle PINN — corrected Cartesian formulation

نسخهٔ یکپارچه: **GPU و Internet را در Kaggle فعال کنید و Run All بزنید.**
به نوت‌بوک قبلی یا فایل جداگانه نیاز ندارد. پیش‌فرض از ابتدا آموزش می‌دهد.

- هندسه ثابت: `R_in=0.2`, `R_out=0.5`.
- PINN جریان: `x,y,Re → u,v,p` با خروجی دکارتی نرم و فشار با مرجع مشخص.
- سرعت مرزی با loss مرز اعمال می‌شود؛ شرط مشتق اضافی نسخهٔ قبلی حذف شده است.
- انتگرال شار گاوس–لژاندر؛ نمودار معیارها روی نقاط ثابت؛ مرحلهٔ نهایی L-BFGS.
- شش پارامتر فیزیکی: `Re,diameter,density_ratio,x_p0,u_p0,v_p0`.
- ذره با ODE یک‌طرفه و درگ Schiller–Naumann رهگیری می‌شود؛ برای تغییر شرایط اولیه آموزش مجدد لازم نیست.
- در پایان، مقایسهٔ میدان با FEM و وب‌اپ انتخاب موقعیت و سرعت اولیه اجرا می‌شوند.

آموزش کامل نسخهٔ اصلاح‌شده روی GPU توسط سازنده اجرا نشده است. پایین‌بودن residual
به‌تنهایی تأیید دقت نیست؛ خطای مرزی و `validation.json` را هم بررسی کنید.


In [ ]:
from pathlib import Path
import sys, subprocess, base64, hashlib, io, zipfile, json, time, math, copy
from types import SimpleNamespace
import numpy as np
from IPython.display import display, FileLink, Image
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
ROOT = BASE / 'particle_corrected_source'
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ZIP = base64.b64decode('...')
assert hashlib.sha256(SOURCE_ZIP).hexdigest() == '42b1f795ca2df39d94869112632a48863adc66355c1cade2e9762e4685d773cf'
with zipfile.ZipFile(io.BytesIO(SOURCE_ZIP)) as z:
    for name in z.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT.resolve()): raise RuntimeError('Invalid source member')
    z.extractall(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
# A previous notebook may have imported the old package. Use this notebook's source consistently.
for name in list(sys.modules):
    if name == 'particle_pinn' or name.startswith('particle_pinn.') or name == 'run_injection_pinn':
        del sys.modules[name]
sys.path.insert(0,str(ROOT))
import torch
print('Source ready:',ROOT,'| CUDA:',torch.cuda.is_available())


## تنظیمات
پیش‌فرض: آموزش تازه با ۱۰هزار گام Adam و سپس حداکثر ۲۰۰ گام داخلی L-BFGS. برای شروع به هیچ فایل قبلی نیاز نیست.


In [ ]:
# All quantities are nondimensional. Edit here only; then Run All.
DEVICE = 'cuda'
RANGES = {
    're': [0.1,10.0],
    'diameter': [0.0025,0.25],
    'density_ratio': [100.0,10000.0],
    'x_p0': [-0.14,0.14],
    'u_p0': [-0.5,0.5],
    'v_p0': [0.25,1.5],
}
WIDTH, DEPTH = 128, 6
SAMPLES = 1024
ADAM_STEPS = 6000
HOURS = 6.0             # Optional cap on the Adam stage. L-BFGS and exports follow.
CHECK_EVERY = 200
LBFGS_ROUNDS = 10
LBFGS_INNER = 20
SEED = 42
TMAX = 10.0
SKIP_FEM = False
# Optional transfer from old weights if they are already available in this session:
PREVIOUS_RUN = None     # e.g. '/kaggle/working/injection_...'; default needs no old model.
DISTILL_STEPS = 1000    # Used only if PREVIOUS_RUN is set.
# Viewer uses one Re/diameter/density at a time; position and speeds are interactive.
VIEW_RE = 1.0
VIEW_DIAMETER = 0.025
VIEW_DENSITY_RATIO = 1000.0


## آموزش و بررسی مستقل
شرایط اولیهٔ ذره وارد رهگیری ODE می‌شوند. مقایسهٔ FEM و تولید مسیر پس از آموزش انجام می‌شوند.


In [ ]:
import particle_pinn.injection as inj
from particle_pinn.smooth_flow_v2 import SmoothCartesianFlow
from particle_pinn.parametric import map_points, coordinates, pde_residual
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in notebook settings, or explicitly set DEVICE to cpu for a short test.')
if min(ADAM_STEPS,SAMPLES,CHECK_EVERY,WIDTH,DEPTH,LBFGS_INNER)<1 or min(LBFGS_ROUNDS,DISTILL_STEPS)<0:
    raise ValueError('Invalid training counts.')
if HOURS is not None and (not np.isfinite(HOURS) or HOURS<=0): raise ValueError('HOURS must be positive or None.')
ranges=inj.ParameterRanges(**{k:tuple(v) for k,v in RANGES.items()}).validate()
seed=SEED
torch.manual_seed(seed)
torch.set_num_threads(2)
rng=np.random.default_rng(seed)
model=SmoothCartesianFlow(WIDTH,DEPTH).to(DEVICE)
teacher=None
if PREVIOUS_RUN is not None:
    oldpath=Path(PREVIOUS_RUN)
    teacher,oldcfg=inj.load_injection_model(oldpath/'best_model.pt',device=DEVICE)
    if ranges.re[0]<oldcfg['ranges']['re'][0] or ranges.re[1]>oldcfg['ranges']['re'][1]:
        raise ValueError('Teacher Reynolds range does not cover the requested range.')
RUN=BASE/('injection_corrected_'+str(time.time_ns()))
RUN.mkdir()
(RUN/'training_source.zip').write_bytes(SOURCE_ZIP)
cfg=dict(scope='fixed-s-channel-smooth-v2',flow_variant='smooth_cartesian_v2',width=WIDTH,depth=DEPTH,
    out=str(RUN),ranges=ranges.json(),seed=seed,steps=ADAM_STEPS,samples=SAMPLES,tmax=TMAX,
    hours=HOURS,distill_steps=DISTILL_STEPS if teacher is not None else 0,
    parent_run=str(PREVIOUS_RUN) if PREVIOUS_RUN is not None else None,
    fixed_geometry=dict(rin=0.2,rout=0.5),physical_parameters=list(inj.PARAMETER_NAMES),
    flow_inputs=['x','y','Re'],coupling='one-way',
    training='Smooth Cartesian PINN + boundary loss + PDE + Gauss flux; FEM held out',
    units='Nondimensional; position / L0, velocity / U0, time / (L0/U0)',
    history_note='Fixed independent PDE and boundary probe; boundary/flux loss included in selection')
(RUN/'config.json').write_text(json.dumps(cfg,indent=2))
print('NEW RUN:',RUN,flush=True)
H=inj.HALF_WIDTH;R=inj.RADIUS
# Fixed geometry for velocity BCs. Differentiation through this map is not used in the PDE.
def boundary(n=128):
    q=torch.linspace(0,1,n,device=DEVICE)
    eta=torch.linspace(-1,1,n,device=DEVICE)
    qr=torch.cat((q,q,torch.zeros_like(eta),torch.ones_like(eta)))
    er=torch.cat((-torch.ones_like(q),torch.ones_like(q),eta,eta))
    re=torch.tensor(np.exp(rng.uniform(np.log(ranges.re[0]),np.log(ranges.re[1]),len(qr))),dtype=torch.float32,device=DEVICE)[:,None]
    p=model.geometry_parameters(re);xy=map_points(qr,er,p).detach()
    uv=torch.zeros((len(qr),2),device=DEVICE)
    uv[2*n:,1]=1-er[2*n:]**2
    return xy,re,uv
bx,br,bt=boundary()
# Gauss-Legendre integrates a quadratic parabolic profile exactly (up to roundoff).
nodes,weights=np.polynomial.legendre.leggauss(24)
et=torch.tensor(nodes,dtype=torch.float32,device=DEVICE)
wt=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
rbank=torch.tensor([ranges.re[0],np.sqrt(np.prod(ranges.re)),ranges.re[1]],dtype=torch.float32,device=DEVICE)
fr=rbank.repeat_interleave(3*len(et))[:,None]
fq=torch.tensor([.2,.5,.8],device=DEVICE).repeat_interleave(len(et)).repeat(3)
fe=et.repeat(9);fp=model.geometry_parameters(fr)
fx=map_points(fq,fe,fp).detach();_,_,ft=coordinates(fx,fp)

# Fixed independent monitoring points shared with the old-model comparison.
mx,mr=inj.sample_flow_batch(np.random.default_rng(seed+809),2048,ranges.re,DEVICE)
mx=mx.detach()
def terms(net,xy,re,boundary_data=None):
    pp=net.geometry_parameters(re)
    res=pde_residual(lambda a,b:net(a,b[:,2:3]),xy,pp)
    c=res[:,0].square().mean();m=res[:,1:].square().mean()
    px,pr,pt=(bx,br,bt) if boundary_data is None else boundary_data
    bc=(net(px,pr)[:,:2]-pt).square().mean()
    ut=(net(fx,fr)[:,:2]*ft).sum(1).reshape(9,len(et))
    flux=((ut*wt).sum(1)/(4/3)-1).square().mean()
    total=20*c+m+100*bc+5*flux
    return total,c,m,bc,flux

def measure(net):
    # Residuals require autograd even during evaluation.
    with torch.enable_grad():
        values=terms(net,mx.clone().requires_grad_(True),mr)
        return [float(v.detach()) for v in values]
old_values=measure(teacher) if teacher is not None else None
print('Previous model on common probe (None = fresh start):',old_values,flush=True)

if teacher is not None:
    opt=torch.optim.Adam(model.parameters(),lr=5e-4)
    for step in range(DISTILL_STEPS):
        xx,rr=inj.sample_flow_batch(rng,512,ranges.re,DEVICE)
        xx=xx.detach()
        with torch.no_grad(): target=teacher(xx,rr)
        pred=model(xx,rr)
        loss=(pred[:,:2]-target[:,:2]).square().mean()+((pred[:,2]-target[:,2])/(4*np.pi*R/H**2)).square().mean()
        loss=loss+5*(model(bx,br)[:,:2]-bt).square().mean()
        opt.zero_grad(set_to_none=True);loss.backward();opt.step()
        if (step+1)%200==0: print('DISTILL',step+1,float(loss.detach()),flush=True)
    teacher.cpu()

# Different output representation: start a fresh optimizer for the corrected physics problem.
opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-6)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=ADAM_STEPS,eta_min=2e-5)
history=[];best_score=float('inf');best=None;best_step=0;started=time.monotonic()
def checkpoint(step,phase):
    global best_score,best,best_step
    model.eval();v=measure(model)
    if not np.isfinite(v).all(): raise FloatingPointError('Non-finite fixed monitor')
    row=dict(step=step,total=v[0],continuity=v[1],momentum=v[2],boundary=v[3],flux=v[4],phase=phase,elapsed_seconds=time.monotonic()-started)
    history.append(row);print(json.dumps(row),flush=True)
    if v[0]<best_score:
        best_score=v[0];best_step=step;best={k:t.detach().cpu().clone() for k,t in model.state_dict().items()}
        torch.save(dict(scope=cfg['scope'],config=cfg,step=best_step,model=best,
                        selection='Lowest fixed total INCLUDING velocity boundary and flux losses'),RUN/'best_model.pt')
    (RUN/'fixed_history.json').write_text(json.dumps(history,indent=2))
checkpoint(0,'initial_or_after_transfer')
train_boundary=boundary()
last_adam_step=0
for step in range(1,ADAM_STEPS+1):
    if HOURS is not None and time.monotonic()-started>=HOURS*3600: break
    if step%20==1: train_boundary=boundary()
    model.train();xx,rr=inj.sample_flow_batch(rng,SAMPLES,ranges.re,DEVICE)
    loss=terms(model,xx,rr,train_boundary)[0]
    if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss; periodic best_model.pt is retained')
    opt.zero_grad(set_to_none=True);loss.backward()
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),10.)
    if not torch.isfinite(norm): raise FloatingPointError('Non-finite gradient')
    opt.step();scheduler.step();last_adam_step=step
    if step%CHECK_EVERY==0 or step==ADAM_STEPS: checkpoint(step,'AdamW')

checkpoint(last_adam_step,'AdamW_final')
# A fixed collocation set is necessary for deterministic L-BFGS line searches.
model.load_state_dict(best)
lx,lr=inj.sample_flow_batch(np.random.default_rng(seed+3441),4096,ranges.re,DEVICE)
lx=lx.detach()
polish_boundary=boundary(256)
polish=torch.optim.LBFGS(model.parameters(),lr=.5,max_iter=LBFGS_INNER,history_size=30,line_search_fn='strong_wolfe')
for outer in range(LBFGS_ROUNDS):
    def closure():
        polish.zero_grad(set_to_none=True)
        loss=terms(model,lx.clone().requires_grad_(True),lr,polish_boundary)[0]
        if not torch.isfinite(loss): raise FloatingPointError('Non-finite L-BFGS loss')
        loss.backward();return loss
    try:
        polish.step(closure)
        checkpoint(last_adam_step+int(polish.state[next(model.parameters())].get('n_iter',0)),'LBFGS_outer_'+str(outer+1))
    except (RuntimeError,FloatingPointError) as error:
        print('L-BFGS stopped:',error,'; retained best finite model',flush=True);break
model.load_state_dict(best)
final_values=measure(model)
vx,vr,vt=boundary(257)  # Independent boundary points/Re, not the training boundary batch.
with torch.no_grad():
    bpred=model(vx,vr)[:,:2]
    boundary_rms=float((bpred-vt).square().mean().sqrt())
    boundary_max=float((bpred-vt).abs().max())
summary=dict(old_common_probe=old_values,new_common_probe=final_values,selected_step=best_step,
    boundary_rms=boundary_rms,boundary_max_component_error=boundary_max,
    note='A lower residual alone does not establish accuracy. Inspect soft-BC errors and independent FEM validation.')
(RUN/'training_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2),flush=True)
model.cpu().eval()
from run_injection_pinn import export_results
export_args=SimpleNamespace(sweep_count=5,particles=12,seed=seed,tmax=TMAX,
    track_samples=260,skip_reference=SKIP_FEM,reference_ns=64,reference_nn=14)
export_failures=[]
try:
    export_failures=export_results(model,RUN,ranges,export_args,history)
except Exception as error:
    import traceback
    export_failures=[dict(stage='export',error=str(error),traceback=traceback.format_exc())]
(RUN/'failures.json').write_text(json.dumps(export_failures,indent=2))
(RUN/'run_complete.json').write_text(json.dumps(dict(adam_steps=last_adam_step,selected_step=best_step,
    elapsed_seconds=time.monotonic()-started,export_failures=len(export_failures),
    note='Completion is not an accuracy guarantee. Inspect FEM and independent boundary errors.'),indent=2))
# Plot all fixed-probe terms, including the boundary error omitted by the old plot.
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4),layout='constrained')
for key in ['total','continuity','momentum','boundary','flux']:
    ax.semilogy([r['step'] for r in history],[max(r[key],1e-16) for r in history],label=key)
ax.set(xlabel='Optimizer iteration (Adam + actual L-BFGS iterations)',ylabel='Fixed probe loss')
ax.legend();fig.savefig(RUN/'fixed_probe_loss.png',dpi=220);plt.close(fig)
print('RESULTS:',RUN,'| export failures:',len(export_failures),flush=True)


In [ ]:
print((RUN/'training_summary.json').read_text())
for name in ['fixed_probe_loss.png','injection_summary.png']:
    if (RUN/name).exists(): display(Image(filename=str(RUN/name),width=1200))
if (RUN/'validation.json').exists(): print((RUN/'validation.json').read_text())
if (RUN/'failures.json').exists(): print('Execution failures:',(RUN/'failures.json').read_text())


## وب‌اپ
موقعیت را روی عرض ورودی انتخاب کنید و برای سرعت اولیه ماوس را بکشید. ساخت جدول سرعت ممکن است کمی زمان ببرد؛ پس از آن رابط در مرورگر کار می‌کند.


In [ ]:
# Paste this entire file into ONE new code cell after the completed training notebook.
from pathlib import Path
import sys, json, html, time
import numpy as np
from IPython.display import HTML, display, FileLink

# Settings for this viewer (all nondimensional). No training is performed.
VIEW_RE = VIEW_RE
VIEW_DIAMETER = VIEW_DIAMETER
VIEW_DENSITY_RATIO = VIEW_DENSITY_RATIO
VIEW_TMAX = TMAX
NQ, NETA = 401, 81  # Increase for a finer velocity lookup table.
if not np.isfinite(VIEW_TMAX) or VIEW_TMAX <= 0 or min(NQ, NETA) < 3:
    raise ValueError('VIEW_TMAX must be positive; NQ and NETA must be at least 3.')

_source = Path(globals().get('ROOT', '/kaggle/working/particle_injection_source'))
if not (_source / 'particle_pinn/injection.py').exists():
    raise FileNotFoundError('Run the source/setup cell of the original notebook first.')
sys.path.insert(0, str(_source))
from particle_pinn.injection import load_injection_model, InjectionCase, HALF_WIDTH, RADIUS, LENGTH
from particle_pinn.parametric import map_points
import torch

_run = Path(globals()['RUN']) if 'RUN' in globals() else None
if _run is None or not (_run / 'best_model.pt').exists():
    _files = list(Path('/kaggle/working').glob('injection_*/best_model.pt'))
    if not _files:
        raise FileNotFoundError('No best_model.pt found. Set RUN to your completed output folder.')
    _run = max(_files, key=lambda p: p.stat().st_mtime).parent
_view_model, _cfg = load_injection_model(_run / 'best_model.pt', device='cpu')
_ranges = _cfg['ranges']
for _key, _val in [('re', VIEW_RE), ('diameter', VIEW_DIAMETER), ('density_ratio', VIEW_DENSITY_RATIO)]:
    if not _ranges[_key][0] <= _val <= _ranges[_key][1]:
        raise ValueError(f'{_key}={_val} is outside the saved range {_ranges[_key]}. Edit viewer settings.')
_clear = HALF_WIDTH - VIEW_DIAMETER / 2 - 1e-7
_xrange = [max(_ranges['x_p0'][0], -_clear), min(_ranges['x_p0'][1], _clear)]
if _xrange[0] > _xrange[1]:
    raise ValueError('No admissible inlet position for this particle diameter.')

_q, _eta = torch.meshgrid(torch.linspace(0, 1, NQ), torch.linspace(-1, 1, NETA), indexing='ij')
_p = _view_model.geometry_parameters(torch.full((_q.numel(), 1), VIEW_RE))
_xy = map_points(_q.flatten(), _eta.flatten(), _p).numpy()
_uv = np.concatenate([_view_model.evaluate(_xy[k:k+2048], VIEW_RE)[:, :2]
                      for k in range(0, len(_xy), 2048)]).reshape(NQ, NETA, 2)
if not np.isfinite(_uv).all():
    raise ValueError('The loaded model produced a non-finite velocity field.')

# Measure lookup interpolation error against the actual trained PINN, not CFD.
_rng = np.random.default_rng(902)
_qq = _rng.uniform(0, 1, 1000); _ee = _rng.uniform(-.99, .99, 1000)
_pp = _view_model.geometry_parameters(torch.full((1000, 1), VIEW_RE))
_testxy = map_points(torch.tensor(_qq, dtype=torch.float32), torch.tensor(_ee, dtype=torch.float32), _pp).numpy()
_truth = _view_model.evaluate(_testxy, VIEW_RE)[:, :2]
_a, _b = _qq*(NQ-1), (_ee+1)*.5*(NETA-1)
_i, _j = np.minimum(_a.astype(int), NQ-2), np.minimum(_b.astype(int), NETA-2)
_f, _g = (_a-_i)[:, None], (_b-_j)[:, None]
_interp = ((1-_f)*(1-_g)*_uv[_i,_j] + _f*(1-_g)*_uv[_i+1,_j]
           + (1-_f)*_g*_uv[_i,_j+1] + _f*_g*_uv[_i+1,_j+1])
_lookup_error = float(np.linalg.norm(_interp-_truth)/max(np.linalg.norm(_truth), 1e-12))
print('Loaded:', _run / 'best_model.pt')
print(f'Lookup vs trained PINN relative L2: {_lookup_error:.3%}; no retraining.')
if _lookup_error > .01:
    print('Lookup error > 1%. Increase NQ and NETA and rerun this viewer cell.')
_D = dict(nq=NQ, ne=NETA, uv=_uv.reshape(-1).round(7).tolist(), re=VIEW_RE,
          diameter=VIEW_DIAMETER, density=VIEW_DENSITY_RATIO, tmax=VIEW_TMAX,
          xr=_xrange, ur=_ranges['u_p0'], vr=_ranges['v_p0'], lookup_error=_lookup_error)

_page = r'''<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Particle launch lab</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#0b1220;color:#e6edf6;font:14px system-ui,sans-serif}
main{max-width:1280px;margin:auto;padding:20px}h2{margin:0 0 8px;font-size:23px}p{color:#aebed2;line-height:1.8}
.layout{display:grid;grid-template-columns:360px 1fr;gap:16px;direction:ltr}.panel{background:#121f32;border:1px solid #26384f;border-radius:16px;padding:16px;direction:rtl}
canvas{width:100%;display:block;touch-action:none}#launch{cursor:crosshair;border-radius:10px;background:#0b1627}#view{height:440px}
label{display:flex;justify-content:space-between;gap:8px;margin:14px 0 4px}output{font:13px monospace;color:#63e4c6;direction:ltr}
input[type=range]{width:100%;direction:ltr;accent-color:#54d9bc}button{border:0;border-radius:9px;padding:10px 15px;background:#39c7ad;color:#071a1b;cursor:pointer;font:inherit}
button.alt{background:#26394f;color:#e6edf6}.row{display:flex;gap:8px;flex-wrap:wrap;margin-top:14px}.status{min-height:44px;margin-top:12px;color:#b9cddd;line-height:1.7}
.small{font-size:12px;color:#9eb0c9;line-height:1.8}#meta{direction:ltr;text-align:left}a{color:#63e4c6}
@media(max-width:800px){.layout{grid-template-columns:1fr}#view{height:340px}}
</style><main>
<h2>آزمایشگاه حرکت ذره</h2><p>روی نوار ورودی کلیک کن و ماوس را بکش؛ طول و جهت پیکان، سرعت اولیه را تعیین می‌کند. با رهاکردن ماوس مسیر محاسبه می‌شود.</p>
<div class="layout"><section class="panel">
<canvas id="launch" width="360" height="280" aria-label="انتخاب نقطه و کشیدن بردار سرعت"></canvas>
<label>موقعیت عرضی ورودی <output id="xo"></output></label><input id="x" type="range" aria-label="موقعیت عرضی">
<label>سرعت افقی اولیه <output id="uo"></output></label><input id="u" type="range" aria-label="سرعت افقی">
<label>سرعت عمودی اولیه <output id="vo"></output></label><input id="v" type="range" aria-label="سرعت عمودی">
<div class="row"><button id="replay">پخش دوباره</button><button class="alt" id="pause">توقف / ادامه</button><button class="alt" id="csv">دریافت مسیر CSV</button></div>
<div id="status" class="status" role="status"></div><div id="meta" class="small"></div>
</section><section class="panel"><canvas id="view" aria-label="کانال و مسیر متحرک ذره"></canvas>
<div class="small">موقعیت شروع روی مقطع ورودی y=0 انتخاب می‌شود. همهٔ مقادیر بی‌بعدند. رنگ زمینه: اندازهٔ سرعت سیال؛ نقطه: ذره با قطر انتخاب‌شده.</div>
<div class="small">نمایش سریع از جدول سرعتِ استخراج‌شده از PINN آموزش‌دیده و حل حرکت در مرورگر استفاده می‌کند. نیرو: درگ Schiller–Naumann؛ پایان مسیر در تماس دیواره یا خروج.</div></section></div></main>
<script>
const D=__DATA__;
// PHYSICS_START
const R=.35,H=.15,L=2*Math.PI*R,clamp=(x,a,b)=>Math.max(a,Math.min(b,x));
function map(q,e){let second=q>.5,a=2*Math.PI*q-(second?Math.PI:0),s=Math.sin(a),c=Math.cos(a);if(q===0||q===.5||q===1)s=0;
return second?[3*R-R*c+H*e*c,-R*s+H*e*s]:[R-R*c-H*e*c,R*s+H*e*s];}
function coords(x,y){let second=x>3*R||(x>=R&&y<0),dx=x-(second?3*R:R),dy=second?-y:y;
let a=Math.PI-Math.atan2(Math.max(dy,0),dx),s=(second?a+Math.PI:a)*R,rad=Math.hypot(dx,y),e=(second?R-rad:rad-R)/H;
if(x<R&&y<0){s=y;e=-x/H;}if(x>3*R&&y>0){s=L+y;e=(x-4*R)/H;}return[s,e];}
function fluid(z){let [s,e]=coords(z[0],z[1]),a=clamp(s/L,0,1)*(D.nq-1),b=(clamp(e,-1,1)+1)*.5*(D.ne-1);
let i=Math.min(Math.floor(a),D.nq-2),j=Math.min(Math.floor(b),D.ne-2),f=a-i,g=b-j,out=[0,0];
for(let k=0;k<2;k++){let p=2*(i*D.ne+j)+k;out[k]=(1-f)*(1-g)*D.uv[p]+f*(1-g)*D.uv[p+2*D.ne]+(1-f)*g*D.uv[p+2]+f*g*D.uv[p+2*D.ne+2];}return out;}
function rate(U,V){let rep=D.re*D.diameter*Math.hypot(U[0]-V[0],U[1]-V[1]),f=rep<1000?1+.15*Math.pow(rep,.687):.44*rep/24;
return 18*f/(D.density*D.re*D.diameter*D.diameter);}
function frozen(z,U,k,h){let decay=Math.exp(-k*h),b=-Math.expm1(-k*h)/k;
return[z[0]+U[0]*h+(z[2]-U[0])*b,z[1]+U[1]*h+(z[3]-U[1])*b,U[0]+(z[2]-U[0])*decay,U[1]+(z[3]-U[1])*decay];}
function step(z,h){let U=fluid(z),mid=frozen(z,U,rate(U,z.slice(2)),h/2),Um=fluid(mid);
return frozen(z,Um,rate(Um,mid.slice(2)),h);}
function integrate(x,u,v){let z=[x,0,u,v],t=0,dt=.002,rows=[[0,...z]],outcome='time_limit',attempts=0;
if(v<0)return{rows,outcome:'backflow_exit'};
while(t<D.tmax){if(++attempts>80000)throw Error('بودجهٔ محاسبه تمام شد؛ زمان نمایش را کمتر کن.');
let h=Math.min(dt,D.tmax-t,.01/Math.max(1,Math.hypot(z[2],z[3])));if(h<1e-12)break;
let coarse=step(z,h),half=step(z,h/2),fine=step(half,h/2),err=0;
for(let k=0;k<4;k++){let scale=(k<2?2e-7:2e-6)+2e-5*Math.max(Math.abs(z[k]),Math.abs(fine[k]));err=Math.max(err,Math.abs(fine[k]-coarse[k])/(3*scale));}
if(!fine.every(Number.isFinite)||!Number.isFinite(err))throw Error('حل حرکت به مقدار نامعتبر رسید.');
if(err>1){dt=h*Math.max(.15,.85*Math.pow(err,-1/3));continue;}
let [s,e]=coords(fine[0],fine[1]),tests=[];
if(H*(1-Math.abs(e))-D.diameter/2<=0)tests.push(['stuck',a=>H*(1-Math.abs(coords(a[0],a[1])[1]))-D.diameter/2]);
if(s>=L)tests.push(['exited',a=>L-coords(a[0],a[1])[0]]);
if(s< -1e-12)tests.push(['backflow_exit',a=>coords(a[0],a[1])[0]]);
if(tests.length){let first=1,kind='';for(let [name,fn] of tests){let lo=0,hi=1;for(let j=0;j<32;j++){let m=(lo+hi)/2,zm=step(step(z,h*m/2),h*m/2);if(fn(zm)>0)lo=m;else hi=m;}
if(hi<=first){first=hi;kind=name;}}
z=step(step(z,h*first/2),h*first/2);t+=h*first;rows.push([t,...z]);outcome=kind;break;}
z=fine;t+=h;rows.push([t,...z]);dt=h*(err>1e-12?clamp(.9*Math.pow(err,-1/3),.5,2):2);}
return{rows,outcome};}
// PHYSICS_END
const $=id=>document.getElementById(id),launch=$('launch'),lc=launch.getContext('2d'),view=$('view'),ctx=view.getContext('2d');
let controls={x:$('x'),u:$('u'),v:$('v')},ranges={x:D.xr,u:D.ur,v:D.vr};
for(let k of ['x','u','v']){controls[k].min=ranges[k][0];controls[k].max=ranges[k][1];controls[k].step=Math.max((ranges[k][1]-ranges[k][0])/2000,1e-7);}
controls.x.value=clamp(0,...D.xr);controls.u.value=clamp(0,...D.ur);controls.v.value=clamp(1,...D.vr);
let trajectory=null,clock=0,last=null,paused=false,pending=0,dragging=false,back=document.createElement('canvas'),transform=null;
const gain=Math.min(100,210/Math.max(.1,Math.max(0,D.vr[1])-Math.min(0,D.vr[0]))),baseY=30+Math.max(0,D.vr[1])*gain;
function values(){return['x','u','v'].map(k=>Number(controls[k].value));}
function arrow(c,x,y,dx,dy,color){let a=Math.atan2(dy,dx);c.strokeStyle=color;c.fillStyle=color;c.lineWidth=2;c.beginPath();c.moveTo(x,y);c.lineTo(x+dx,y+dy);c.stroke();c.beginPath();c.moveTo(x+dx,y+dy);c.lineTo(x+dx-9*Math.cos(a-.45),y+dy-9*Math.sin(a-.45));c.lineTo(x+dx-9*Math.cos(a+.45),y+dy-9*Math.sin(a+.45));c.fill();}
function launchDraw(){let[x,u,v]=values();$('xo').textContent=x.toFixed(4);$('uo').textContent=u.toFixed(4);$('vo').textContent=v.toFixed(4);
lc.clearRect(0,0,360,280);let a=180+950*D.xr[0],b=180+950*D.xr[1];lc.strokeStyle='#526b88';lc.lineWidth=7;lc.beginPath();lc.moveTo(a,baseY);lc.lineTo(b,baseY);lc.stroke();
lc.fillStyle='#aebed2';lc.font='12px sans-serif';lc.textAlign='center';lc.fillText('x_p0   •   ورودی کانال',180,Math.min(273,baseY+35));
let px=180+950*x;arrow(lc,px,baseY,gain*u,-gain*v,'#63e4c6');lc.fillStyle='#ffcd75';lc.beginPath();lc.arc(px,baseY,6,0,7);lc.fill();}
function pathWalls(c){for(let e of [-1,1]){c.beginPath();for(let i=0;i<=500;i++){let p=transform(map(i/500,e));if(i===0)c.moveTo(...p);else c.lineTo(...p);}c.stroke();}}
function resize(){let w=Math.max(300,view.clientWidth),h=view.clientHeight||440,dpr=Math.min(2,window.devicePixelRatio||1);view.width=w*dpr;view.height=h*dpr;ctx.setTransform(dpr,0,0,dpr,0,0);
back.width=view.width;back.height=view.height;let c=back.getContext('2d');c.setTransform(dpr,0,0,dpr,0,0);
let scale=Math.min((w-50)/1.85,(h-70)/1.10),ox=(w-scale*1.4)/2,oy=h/2;transform=p=>[ox+p[0]*scale,oy-p[1]*scale];transform.scale=scale;
c.fillStyle='#0c1728';c.fillRect(0,0,w,h);let vmax=0;for(let i=0;i<D.uv.length;i+=2)vmax=Math.max(vmax,Math.hypot(D.uv[i],D.uv[i+1]));
for(let i=0;i<D.nq-1;i+=2)for(let j=0;j<D.ne-1;j+=2){let k=2*(i*D.ne+j),s=Math.hypot(D.uv[k],D.uv[k+1])/Math.max(vmax,1e-12);
c.fillStyle=`hsl(${225-65*s},65%,${16+22*s}%)`;let corners=[[i,j],[Math.min(i+2,D.nq-1),j],[Math.min(i+2,D.nq-1),Math.min(j+2,D.ne-1)],[i,Math.min(j+2,D.ne-1)]];
c.beginPath();corners.forEach(([ii,jj],n)=>{let p=transform(map(ii/(D.nq-1),2*jj/(D.ne-1)-1));if(!n)c.moveTo(...p);else c.lineTo(...p);});c.closePath();c.fill();}
c.strokeStyle='#8fa6c2';c.lineWidth=1.2;pathWalls(c);c.fillStyle='#bdcce0';c.font='12px sans-serif';let p=transform([0,0]);c.fillText('inlet',p[0]-15,p[1]+25);
p=transform([4*R,0]);c.fillText('outlet',p[0]-15,p[1]-15);draw();}
function draw(){if(!transform)return;let dpr=Math.min(2,window.devicePixelRatio||1);ctx.clearRect(0,0,view.clientWidth,view.clientHeight);ctx.drawImage(back,0,0,back.width/dpr,back.height/dpr);
let[x,u,v]=values(),p=transform([x,0]);arrow(ctx,...p,u*45,-v*45,'#63e4c6');
if(!trajectory)return;let rows=trajectory.rows;ctx.strokeStyle='#ffcd7566';ctx.lineWidth=1.5;ctx.beginPath();rows.forEach((r,i)=>{let p=transform(r.slice(1,3));if(!i)ctx.moveTo(...p);else ctx.lineTo(...p);});ctx.stroke();
let i=0;while(i<rows.length-2&&rows[i+1][0]<clock)i++;let a=rows[i],b=rows[Math.min(i+1,rows.length-1)],f=b[0]>a[0]?clamp((clock-a[0])/(b[0]-a[0]),0,1):0;
let z=[a[1]+f*(b[1]-a[1]),a[2]+f*(b[2]-a[2])];p=transform(z);ctx.fillStyle='#ffcd75';ctx.beginPath();ctx.arc(...p,Math.max(2.5,D.diameter*transform.scale/2),0,7);ctx.fill();
ctx.fillStyle='#dae8f7';ctx.font='13px monospace';ctx.fillText('t* = '+Math.min(clock,rows[rows.length-1][0]).toFixed(3),18,24);}
function replay(){clock=0;last=null;paused=false;draw();}
function compute(){clearTimeout(pending);$('status').textContent='در حال محاسبه…';setTimeout(()=>{try{let start=performance.now();trajectory=integrate(...values());replay();
let names={stuck:'تماس با دیواره',exited:'خروج از کانال',backflow_exit:'خروج از ورودی',time_limit:'پایان زمان محاسبه'};
$('status').textContent=names[trajectory.outcome]+' | زمان محاسبه: '+Math.round(performance.now()-start)+' ms';}catch(e){trajectory=null;$('status').textContent=e.message;}draw();},20);}
for(let k of ['x','u','v'])controls[k].addEventListener('input',()=>{launchDraw();trajectory=null;draw();clearTimeout(pending);pending=setTimeout(compute,160);});
function pointer(ev){let b=launch.getBoundingClientRect();return[(ev.clientX-b.left)*360/b.width,(ev.clientY-b.top)*280/b.height];}
launch.addEventListener('pointerdown',ev=>{let[p,y]=pointer(ev),[x,u,v]=values(),tip=[180+950*x+gain*u,baseY-gain*v];
if(Math.abs(y-baseY)<25)controls.x.value=clamp((p-180)/950,...D.xr);else if(Math.hypot(p-tip[0],y-tip[1])>25)return;
dragging=true;launch.setPointerCapture(ev.pointerId);clearTimeout(pending);trajectory=null;launchDraw();draw();});
launch.addEventListener('pointermove',ev=>{if(!dragging)return;let[p,y]=pointer(ev);controls.u.value=clamp((p-180-950*Number(controls.x.value))/gain,...D.ur);controls.v.value=clamp((baseY-y)/gain,...D.vr);launchDraw();draw();});
function release(){if(dragging){dragging=false;compute();}}launch.addEventListener('pointerup',release);launch.addEventListener('pointercancel',release);
$('replay').onclick=replay;$('pause').onclick=()=>{paused=!paused;last=null;};
$('csv').onclick=()=>{if(!trajectory)return;let[x,u,v]=values(),text=`# Re=${D.re},diameter=${D.diameter},density_ratio=${D.density},x_p0=${x},u_p0=${u},v_p0=${v},outcome=${trajectory.outcome}\nt,x,y,u_p,v_p\n`+trajectory.rows.map(r=>r.join(',')).join('\n');
let url=URL.createObjectURL(new Blob([text],{type:'text/csv'})),a=document.createElement('a');a.href=url;a.download='particle_trajectory.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};
$('meta').textContent=`Re=${D.re} | d_p=${D.diameter} | rho_p/rho_f=${D.density}\nLookup L2 vs PINN: ${(100*D.lookup_error).toFixed(3)}%`;
function tick(now){if(last!==null&&!paused&&trajectory)clock=Math.min(trajectory.rows[trajectory.rows.length-1][0],clock+Math.min(.1,(now-last)/1000));last=now;draw();requestAnimationFrame(tick);}
new ResizeObserver(resize).observe(view);launchDraw();resize();compute();requestAnimationFrame(tick);
</script></html>'''
_page = _page.replace('__DATA__', json.dumps(_D, separators=(',', ':'), allow_nan=False))
_app_path = _run.parent / 'particle_live_app.html'
_app_path.write_text(_page, encoding='utf-8')
display(HTML('<iframe title="Particle launch lab" style="width:100%;height:880px;border:0;border-radius:16px" '
             'sandbox="allow-scripts allow-downloads" srcdoc="'+html.escape(_page, quote=True)+'"></iframe>'))
print('Standalone app — also downloadable from the Kaggle Output panel:')
display(FileLink(str(_app_path)))


## دریافت خروجی‌ها
فایل HTML را می‌توانید پس از دانلود بدون روشن‌بودن Kaggle باز کنید.


In [ ]:
import shutil
archive=shutil.make_archive(str(RUN),'zip',root_dir=RUN)
print('All results, weights and training source:')
display(FileLink(archive))
print('Standalone web app:')
display(FileLink(str(_app_path)))


In [ ]:
# ONE NEW CELL after the completed training notebook. Run this cell only.
# No neural training: optimize inlet controls and release timing, verify with frozen PINN.
# Extension flow is explicitly assumed planar Poiseuille. All quantities are nondimensional.
from pathlib import Path
from dataclasses import asdict
import sys, importlib.util, json, csv, time, math, html, shutil
import numpy as np
import torch
from IPython.display import display, HTML, FileLink

MODEL_RUN = None  # None uses RUN from the completed notebook.
FLOW_RE = 1.0
PARTICLE_DIAMETER = 0.025
PARTICLE_DENSITY_RATIO = 1000.0
EXTENSION_LENGTH = 1.4  # 2 x centreline bend diameter (2 x 0.7), same channel width 0.3.
TARGET_RADIUS = 0.075  # Circumradius of equilateral triangle / square.
POSITION_TOLERANCE = 0.001
SEARCH_BANK_POWER = 6  # 2**6 + 11 seed launches, shared by both shapes.
LOCAL_STARTS = 3
SHOOTING_ITERATIONS = 12
DIRECT_PINN_ITERATIONS = 6
CHANNEL_TMAX = 20.0
EXTENSION_TMAX = 20.0
COLLISION_MARGIN = 0.001
NQ, NETA = 401, 81  # Velocity lookup for fast search; final paths use the real PINN.

_fc_sources=[Path(globals().get('ROOT','/kaggle/working/particle_corrected_source')),
             Path('/kaggle/working/particle_corrected_source'),Path('/kaggle/working/particle_injection_source')]
_fc_source=next((p for p in _fc_sources if (p/'particle_pinn/injection.py').exists()),None)
if _fc_source is None: raise FileNotFoundError('Run the setup cell of the training notebook first.')
sys.path.insert(0,str(_fc_source))
from particle_pinn.injection import load_injection_model, HALF_WIDTH, RADIUS
from particle_pinn.parametric import map_points
_fc_run=Path(MODEL_RUN) if MODEL_RUN is not None else Path(globals()['RUN']) if 'RUN' in globals() else None
if _fc_run is None or not (_fc_run/'best_model.pt').exists():
    if MODEL_RUN is not None:raise FileNotFoundError('MODEL_RUN does not contain best_model.pt')
    _fc_candidates=list(Path('/kaggle/working').glob('injection_*/best_model.pt'))
    if not _fc_candidates:raise FileNotFoundError('No best_model.pt found. Set MODEL_RUN explicitly.')
    _fc_run=max(_fc_candidates,key=lambda p:p.stat().st_mtime).parent
_fc_model,_fc_cfg=load_injection_model(_fc_run/'best_model.pt',device='cpu')
_fc_model.eval().requires_grad_(False)
torch.set_num_threads(1)
for _k,_v in [('re',FLOW_RE),('diameter',PARTICLE_DIAMETER),('density_ratio',PARTICLE_DENSITY_RATIO)]:
    if not np.isfinite(_v) or not _fc_cfg['ranges'][_k][0]<=_v<=_fc_cfg['ranges'][_k][1]:
        raise ValueError(f'{_k}={_v} is outside the saved physical range')
if min(EXTENSION_LENGTH,TARGET_RADIUS,POSITION_TOLERANCE,CHANNEL_TMAX,EXTENSION_TMAX)<=0 or not np.isfinite([EXTENSION_LENGTH,TARGET_RADIUS,POSITION_TOLERANCE,CHANNEL_TMAX,EXTENSION_TMAX,COLLISION_MARGIN]).all() or COLLISION_MARGIN<0:
    raise ValueError('Invalid formation settings')
if not 0<=SEARCH_BANK_POWER<=10 or not 1<=LOCAL_STARTS<=8 or min(SHOOTING_ITERATIONS,DIRECT_PINN_ITERATIONS)<1 or min(NQ,NETA)<3:
    raise ValueError('Invalid optimization or lookup budget')
_fc_clear=HALF_WIDTH-PARTICLE_DIAMETER/2-1e-6
_fc_bounds=[[max(_fc_cfg['ranges']['x_p0'][0],-_fc_clear),min(_fc_cfg['ranges']['x_p0'][1],_fc_clear)],
            list(_fc_cfg['ranges']['u_p0']),list(_fc_cfg['ranges']['v_p0'])]
if _fc_bounds[0][0]>_fc_bounds[0][1]:raise ValueError('No admissible inlet position for this particle')
_fc_out=_fc_run/('formation_control_'+str(time.time_ns()));_fc_out.mkdir()
print('Frozen model:',_fc_run/'best_model.pt',flush=True)
print('Output:',_fc_out,flush=True)

# Export the physical search engine beside the results, without altering model source/weights.
_FC_CORE = r'''"""Inverse release control in a frozen steady flow; no neural training."""
from dataclasses import dataclass
from itertools import product
import math,time
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar
from scipy.stats import qmc

R=.35; H=.15; LENGTH=2*math.pi*R; XC=4*R

def coordinates(x,y):
    second=x>3*R or (x>=R and y<0)
    dx=x-(3*R if second else R);dy=-y if second else y
    a=math.pi-math.atan2(max(dy,0.),dx)
    s=(a+math.pi if second else a)*R
    rad=math.hypot(dx,y);eta=(R-rad if second else rad-R)/H
    if x<R and y<0:s=y;eta=-x/H
    if x>3*R and y>0:s=LENGTH+y;eta=(x-XC)/H
    return s,eta

def map_point(q,e):
    second=q>.5;a=2*math.pi*q-(math.pi if second else 0.)
    si=math.sin(a);co=math.cos(a)
    if q in (0.,.5,1.):si=0.
    return (3*R-R*co+H*e*co,-R*si+H*e*si) if second else (R-R*co-H*e*co,R*si+H*e*si)

@dataclass
class Settings:
    re:float=1.
    diameter:float=.025
    density:float=1000.
    extension:float=1.4
    target_radius:float=.075
    target_tol:float=.001
    channel_tmax:float=20.
    extension_tmax:float=20.
    bank_power:int=6
    starts:int=3
    iterations:int=12
    direct_iterations:int=6
    collision_margin:float=.001
    lead_time:float=.5
    seed:int=42

class Trajectory:
    def __init__(self,control,channel,extension,status):
        self.control=np.array(control,float);self.channel=channel;self.extension=extension;self.status=status
    def hit(self,target_y):
        if self.extension is None:return None
        sol=self.extension
        # First upward crossing of the desired y, using dense output and root finding.
        crossings=np.flatnonzero((sol.y[1,:-1]<=target_y)&(sol.y[1,1:]>=target_y))
        if not len(crossings):return None
        i=crossings[0]
        tau=brentq(lambda t:sol.sol(t)[1]-target_y,sol.t[i],sol.t[i+1],xtol=1e-11)
        state=sol.sol(tau)
        return dict(age=float(self.channel.t[-1]+tau),state=state)
    def state(self,ages):
        ages=np.atleast_1d(ages);out=np.empty((len(ages),4));tc=self.channel.t[-1]
        before=ages<=tc
        if np.any(before):out[before]=self.channel.sol(np.clip(ages[before],0,tc)).T
        if np.any(~before):out[~before]=self.extension.sol(ages[~before]-tc).T
        return out

class Planner:
    def __init__(self,field,settings,bounds,tag='lookup'):
        self.field=field;self.c=settings;self.bounds=np.array(bounds,float);self.tag=tag;self.cache={};self.evaluations=0
        if self.bounds.shape!=(3,2) or np.any(self.bounds[:,0]>self.bounds[:,1]):raise ValueError('Invalid control bounds')
        self.span=self.bounds[:,1]-self.bounds[:,0]
    def physical(self,z):return self.bounds[:,0]+np.clip(z,0,1)*self.span
    def normalized(self,u):return np.divide(np.asarray(u)-self.bounds[:,0],self.span,out=np.zeros(3),where=self.span>0)
    def _acceleration(self,fluid,z):
        du,dv=fluid[0]-z[2],fluid[1]-z[3]
        rep=self.c.re*self.c.diameter*math.hypot(du,dv)
        f=1+.15*rep**.687 if rep<1000 else .44*rep/24
        k=18*f/(self.c.density*self.c.re*self.c.diameter**2)
        return du*k,dv*k
    def path(self,z):
        control=self.physical(z);key=tuple(np.round(control,11))
        if key in self.cache:return self.cache[key]
        self.evaluations+=1;c=self.c;x0,u0,v0=control;evals=0
        radius=c.diameter/2
        if abs(x0)+radius>=H or v0<0:
            result=Trajectory(control,None,None,'invalid_injection');self.cache[key]=result;return result
        def rhs(t,state):
            nonlocal evals
            evals+=1
            if evals>30000:raise RuntimeError('Channel RHS budget exceeded')
            sx,et=coordinates(state[0],state[1])
            # Only event-bracketing stages may leave the domain: query a boundary
            # projection there, never interpret PINN extrapolation as an extension.
            fx,fy=map_point(min(1.,max(0.,sx/LENGTH)),min(1.,max(-1.,et))) if (sx<0 or sx>LENGTH or abs(et)>1) else (state[0],state[1])
            fluid=self.field(fx,fy)
            if not np.isfinite(fluid).all():raise FloatingPointError('Non-finite velocity')
            ax,ay=self._acceleration(fluid,state)
            return [state[2],state[3],ax,ay]
        def wall(t,z):return H*(1-abs(coordinates(z[0],z[1])[1]))-radius
        def outlet(t,z):return LENGTH-coordinates(z[0],z[1])[0]
        def inlet(t,z):return 1e-12 if t==0 and v0>=0 else coordinates(z[0],z[1])[0]
        for ev in [wall,outlet,inlet]:ev.terminal=True;ev.direction=-1
        tau=c.density*c.re*c.diameter**2/18
        method='Radau' if tau<.005 else 'RK45'
        step=min(.025,H/(10*max(1,math.hypot(u0,v0))))
        try:
            ch=solve_ivp(rhs,(0,c.channel_tmax),[x0,0,u0,v0],events=[wall,outlet,inlet],
                dense_output=True,rtol=2e-6,atol=2e-8,max_step=step,method=method)
            if not ch.success:status='channel_solver_failed'
            elif len(ch.t_events[0]):status='channel_wall'
            elif len(ch.t_events[2]):status='inlet_exit'
            elif not len(ch.t_events[1]):status='channel_time_limit'
            else:status='outlet'
            ex=None
            if status=='outlet':
                start=ch.y[:,-1].copy();start[1]=0. # exact cap; position/velocity otherwise preserved
                if abs(start[0]-XC)+radius>=H:status='extension_initial_wall'
                else:
                    evals=0
                    def erhs(t,z):
                        nonlocal evals
                        evals+=1
                        if evals>30000:raise RuntimeError('Extension RHS budget exceeded')
                        fluid=(0.,max(0.,1-((z[0]-XC)/H)**2))
                        ax,ay=self._acceleration(fluid,z)
                        return [z[2],z[3],ax,ay]
                    def ewall(t,z):return H-abs(z[0]-XC)-radius
                    def etop(t,z):return c.extension-z[1]
                    def eback(t,z):return 1e-12 if t==0 and start[3]>=0 else z[1]
                    for ev in [ewall,etop,eback]:ev.terminal=True;ev.direction=-1
                    ex=solve_ivp(erhs,(0,c.extension_tmax),start,events=[ewall,etop,eback],
                        dense_output=True,rtol=2e-7,atol=2e-9,max_step=step,method=method)
                    status='extension_solver_failed' if not ex.success else 'extension_wall' if len(ex.t_events[0]) else 'extension_end' if len(ex.t_events[1]) else 'extension_backflow' if len(ex.t_events[2]) else 'extension_time_limit'
            result=Trajectory(control,ch,ex,status)
        except (RuntimeError,ValueError,FloatingPointError) as error:
            result=Trajectory(control,None,None,'failed: '+str(error))
        self.cache[key]=result
        return result
    def bank(self):
        samples=qmc.Sobol(d=3,scramble=True,seed=self.c.seed).random_base2(self.c.bank_power)
        base=np.array([0.,0.,1.]);base=np.clip(base,self.bounds[:,0],self.bounds[:,1])
        anchors=np.tile(self.normalized(base),(11,1));anchors[:,0]=np.linspace(.001,.999,11)
        samples=np.vstack((anchors,samples))
        bank=[];started=time.monotonic()
        for i,z in enumerate(samples):
            p=self.path(z)
            if p.extension is not None and p.extension.success:bank.append((z,p))
            if (i+1)%16==0:print(f'{self.tag}: search {i+1}/{len(samples)}, exit paths {len(bank)}, {time.monotonic()-started:.1f}s',flush=True)
        return bank
    def refine(self,z,target,iterations=None):
        z=np.clip(np.asarray(z,float),0,1);best=None;trace=[]
        for it in range(self.c.iterations if iterations is None else iterations):
            path=self.path(z);hit=path.hit(target[1])
            if hit is None:break
            error=float(hit['state'][0]-target[0])
            if best is None or abs(error)<best['error']:
                best=dict(z=z.copy(),path=path,hit=hit,error=abs(error))
            trace.append(abs(error))
            if abs(error)<=self.c.target_tol*.15:break
            # Absolute normalized perturbations avoid noisy near-zero relative differences.
            jac=np.zeros(3);delta=.002
            for j in range(3):
                if self.span[j]==0:continue
                zp=z.copy();zm=z.copy();zp[j]=min(1,z[j]+delta);zm[j]=max(0,z[j]-delta)
                hp=self.path(zp).hit(target[1]);hm=self.path(zm).hit(target[1])
                if hp is not None and hm is not None:
                    jac[j]=(hp['state'][0]-hm['state'][0])/(zp[j]-zm[j])
                elif hp is not None and zp[j]>z[j]:jac[j]=(hp['state'][0]-hit['state'][0])/(zp[j]-z[j])
                elif hm is not None and zm[j]<z[j]:jac[j]=(hit['state'][0]-hm['state'][0])/(z[j]-zm[j])
            norm=float(jac@jac)
            if norm<1e-14:break
            move=-error*jac/(norm+1e-10)
            move*=min(1.,.18/max(np.max(np.abs(move)),1e-15))
            improved=False
            for factor in [1.,.5,.25,.125,.0625]:
                trial=np.clip(z+factor*move,0,1);h=self.path(trial).hit(target[1])
                if h is not None and abs(h['state'][0]-target[0])<abs(error):
                    z=trial;improved=True;break
            if not improved:break
        last_path=self.path(z);last_hit=last_path.hit(target[1])
        if last_hit is not None:
            error=abs(float(last_hit['state'][0]-target[0]))
            if best is None or error<best['error']:best=dict(z=z.copy(),path=last_path,hit=last_hit,error=error)
        if best is not None:best['optimization_errors']=trace
        return best


def targets(kind,c):
    n=3 if kind=='triangle' else 4 if kind=='square' else None
    if n is None:raise ValueError('Only triangle and square are implemented')
    phase=math.pi/2 if n==3 else math.pi/4
    angle=phase+np.arange(n)*2*math.pi/n
    pts=np.column_stack((XC+c.target_radius*np.cos(angle),c.extension/2+c.target_radius*np.sin(angle)))
    if np.any(abs(pts[:,0]-XC)+c.diameter/2>=H) or np.any(pts[:,1]-c.diameter/2<=0) or np.any(pts[:,1]+c.diameter/2>=c.extension):
        raise ValueError('Target does not fit the extension with particle-radius clearance')
    distances=np.linalg.norm(pts[:,None,:]-pts[None,:,:],axis=2)+np.eye(n)*1000
    if distances.min()<c.diameter+c.collision_margin:raise ValueError('Target vertices are too close for the particle size')
    return pts


def separation(candidates,c,refine=False):
    ages=np.array([a['hit']['age'] for a in candidates]);T=float(ages.max()+c.lead_time);release=T-ages
    minimum=float('inf')
    for i in range(len(candidates)):
        for j in range(i+1,len(candidates)):
            low=max(release[i],release[j]);grid=np.linspace(low,T,161)
            def distance(t):
                a=candidates[i]['path'].state(np.atleast_1d(t)-release[i])[:,:2]
                b=candidates[j]['path'].state(np.atleast_1d(t)-release[j])[:,:2]
                return np.linalg.norm(a-b,axis=1)
            d=distance(grid);minimum=min(minimum,float(d.min()))
            if refine:
                minima=np.flatnonzero((d[1:-1]<=d[:-2])&(d[1:-1]<=d[2:]))+1
                for k in minima:
                    r=minimize_scalar(lambda t:float(distance(t)[0]),bounds=(grid[k-1],grid[k+1]),method='bounded',options={'xatol':1e-8})
                    minimum=min(minimum,float(r.fun))
    return T,release,minimum


def plan_shape(kind,coarse,direct,bank):
    c=coarse.c;goal=targets(kind,c);options=[];diagnostics=[]
    for index,target in enumerate(goal):
        pool=[]
        for z,p in bank:
            hit=p.hit(target[1])
            if hit is not None:pool.append((abs(hit['state'][0]-target[0]),z))
        pool.sort(key=lambda item:item[0])
        if not pool:
            return dict(kind=kind,success=False,reason=f'No feasible search trajectory reaches target height for vertex {index+1}',targets=goal.tolist(),details=diagnostics)
        candidates=[]
        for _,z in pool[:c.starts]:
            best=coarse.refine(z,target)
            if best is None:continue
            # Verify/refine using the real frozen PINN, not the lookup table.
            verified=direct.refine(best['z'],target,c.direct_iterations)
            if verified is not None and all(np.linalg.norm(verified['z']-p['z'])>1e-5 for p in candidates):candidates.append(verified)
        candidates.sort(key=lambda b:b['error'])
        diagnostics.append(dict(vertex=index+1,available_seeds=len(pool),verified_candidates=len(candidates)))
        print(f'{kind} vertex {index+1}: '+('best direct error %.6g'%candidates[0]['error'] if candidates else 'no verified path'),flush=True)
        if not candidates:
            return dict(kind=kind,success=False,reason=f'No direct-PINN verified path for vertex {index+1}',targets=goal.tolist(),details=diagnostics)
        options.append(candidates[:c.starts])
    ranked=[]
    for combo in product(*options):
        T,release,sep=separation(combo,c)
        error=max(p['error'] for p in combo)
        key=(error>c.target_tol,sep<c.diameter+c.collision_margin,error,T)
        ranked.append((key,combo))
    ranked.sort(key=lambda item:item[0])
    chosen=None;score=None
    # Check alternatives if a narrow close approach was missed by coarse sampling.
    for _,candidate in ranked:
        T,release,sep=separation(candidate,c,refine=True)
        error=max(p['error'] for p in candidate)
        key=(error>c.target_tol,sep<c.diameter+c.collision_margin,error,T)
        if score is None or key<score:score=key;chosen=(candidate,T,release,sep)
        if error<=c.target_tol and sep>=c.diameter+c.collision_margin:break
    combo,T,release,sep=chosen
    actual=np.array([p['hit']['state'][:2] for p in combo]);errors=np.linalg.norm(actual-goal,axis=1)
    schedules=[];paths=[]
    for i,p in enumerate(combo):
        xp,up,vp=p['path'].control;age=p['hit']['age']
        schedules.append(dict(particle=i+1,x_p0=float(xp),u_p0=float(up),v_p0=float(vp),release_time=float(release[i]),
            age_at_target=float(age),target_x=float(goal[i,0]),target_y=float(goal[i,1]),actual_x=float(actual[i,0]),actual_y=float(actual[i,1]),error=float(errors[i])))
        ts=np.linspace(0,age,700);state=p['path'].state(ts)
        paths.append(dict(t=ts.tolist(),state=state.tolist(),release=float(release[i])))
    success=bool(errors.max()<=c.target_tol and sep>=c.diameter+c.collision_margin)
    return dict(kind=kind,success=success,reason='target_and_clearance_met' if success else 'best_found_solution_misses_target_or_clearance',
        target_time=T,targets=goal.tolist(),actual=actual.tolist(),max_error=float(errors.max()),rms_error=float(np.sqrt(np.mean(errors**2))),
        minimum_pair_distance=sep,required_pair_distance=c.diameter+c.collision_margin,
        schedules=schedules,paths=paths,details=diagnostics,
        collision_check='Dense time sampling plus bounded refinement of observed local distance minima; no particle contact forces modeled')
'''
(_fc_out/'formation_core.py').write_text(_FC_CORE,encoding='utf-8')
_fc_spec=importlib.util.spec_from_file_location('particle_formation_core',_fc_out/'formation_core.py')
_fc=importlib.util.module_from_spec(_fc_spec);sys.modules[_fc_spec.name]=_fc;_fc_spec.loader.exec_module(_fc)

_fc_settings=_fc.Settings(re=float(FLOW_RE),diameter=float(PARTICLE_DIAMETER),density=float(PARTICLE_DENSITY_RATIO),
    extension=float(EXTENSION_LENGTH),target_radius=float(TARGET_RADIUS),target_tol=float(POSITION_TOLERANCE),
    channel_tmax=float(CHANNEL_TMAX),extension_tmax=float(EXTENSION_TMAX),bank_power=int(SEARCH_BANK_POWER),
    starts=int(LOCAL_STARTS),iterations=int(SHOOTING_ITERATIONS),direct_iterations=int(DIRECT_PINN_ITERATIONS),
    collision_margin=float(COLLISION_MARGIN),seed=int(_fc_cfg.get('seed',42)))
for _shape in ['triangle','square']:_fc.targets(_shape,_fc_settings)

def _fc_velocity_batch(points):
    with torch.no_grad():
        # For v2 the actual velocity is the raw two-component output; avoid the
        # unnecessary second pressure-gauge pass on each scalar ODE query.
        if _fc_cfg.get('flow_variant')=='smooth_cartesian_v2':
            x=torch.as_tensor(np.atleast_2d(points),dtype=torch.float32)
            rr=x.new_full((len(x),1),FLOW_RE)
            p=_fc_model.geometry_parameters(rr)
            return _fc_model.net(_fc_model.features(x,p))[:,:2].cpu().numpy()
        return _fc_model.evaluate(np.atleast_2d(points),FLOW_RE)[:,:2]

def _fc_direct(x,y):return _fc_velocity_batch([[x,y]])[0]

_q,_e=torch.meshgrid(torch.linspace(0,1,NQ),torch.linspace(-1,1,NETA),indexing='ij')
_pp=_fc_model.geometry_parameters(torch.full((_q.numel(),1),FLOW_RE))
_xy=map_points(_q.flatten(),_e.flatten(),_pp).numpy()
_fc_uv=np.concatenate([_fc_velocity_batch(_xy[k:k+2048]) for k in range(0,len(_xy),2048)]).reshape(NQ,NETA,2)
if not np.isfinite(_fc_uv).all():raise FloatingPointError('Non-finite PINN velocity')
def _fc_lookup(x,y):
    s,e=_fc.coordinates(x,y)
    a=min(1.,max(0.,s/_fc.LENGTH))*(NQ-1);b=(min(1.,max(-1.,e))+1)*.5*(NETA-1)
    i=min(int(a),NQ-2);j=min(int(b),NETA-2);f=a-i;g=b-j
    return (1-f)*(1-g)*_fc_uv[i,j]+f*(1-g)*_fc_uv[i+1,j]+(1-f)*g*_fc_uv[i,j+1]+f*g*_fc_uv[i+1,j+1]

_frng=np.random.default_rng(789)
_qq=_frng.uniform(0,1,1200);_ee=_frng.uniform(-.99,.99,1200)
_testxy=np.array([_fc.map_point(float(q),float(e)) for q,e in zip(_qq,_ee)])
_truth=_fc_velocity_batch(_testxy);_approx=np.array([_fc_lookup(float(x),float(y)) for x,y in _testxy])
_fc_lookup_error=float(np.linalg.norm(_approx-_truth)/max(np.linalg.norm(_truth),1e-12))
# Compare the trained cap velocity with the nominal analytic profile used in the extension.
_xcap=np.linspace(_fc.XC-HALF_WIDTH,_fc.XC+HALF_WIDTH,101)
_cap=_fc_velocity_batch(np.column_stack((_xcap,np.zeros_like(_xcap))))
_nominal=np.column_stack((np.zeros_like(_xcap),1-((_xcap-_fc.XC)/HALF_WIDTH)**2))
_fc_cap_error=float(np.sqrt(np.mean((_cap-_nominal)**2)))
print(f'Lookup velocity relative L2 vs PINN: {_fc_lookup_error:.3%}',flush=True)
print(f'Outlet PINN / assumed-extension velocity RMS mismatch: {_fc_cap_error:.6g}',flush=True)
print('Extension assumption: u=0, v=1-((x-1.4)/0.15)^2; no PINN extrapolation or new CFD solve.',flush=True)
_fc_coarse=_fc.Planner(_fc_lookup,_fc_settings,_fc_bounds,'fast search')
_fc_verify=_fc.Planner(_fc_direct,_fc_settings,_fc_bounds,'direct PINN verification')
_fc_bank=_fc_coarse.bank()
_fc_results={}
for _shape in ['triangle','square']:
    _start=time.monotonic()
    try:_result=_fc.plan_shape(_shape,_fc_coarse,_fc_verify,_fc_bank)
    except Exception as error:
        import traceback
        _result=dict(kind=_shape,success=False,reason=str(error),traceback=traceback.format_exc(),targets=_fc.targets(_shape,_fc_settings).tolist())
    _result['optimization_seconds']=time.monotonic()-_start
    _fc_results[_shape]=_result
    (_fc_out/(_shape+'_result.json')).write_text(json.dumps(_result,indent=2,allow_nan=False))
    _rows=_result.get('schedules',[])
    if _rows:
        with (_fc_out/(_shape+'_release_schedule.csv')).open('w',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=list(_rows[0]));writer.writeheader();writer.writerows(_rows)
        for i,p in enumerate(_result['paths']):
            arr=np.column_stack((p['t'],np.array(p['t'])+p['release'],p['state']))
            np.savetxt(_fc_out/f'{_shape}_particle_{i+1}.csv',arr,delimiter=',',comments='',header='age,global_time,x,y,u_p,v_p')
    print(_shape, '| success:',_result['success'],'| max error:',_result.get('max_error'),
          '| minimum pair distance:',_result.get('minimum_pair_distance'),flush=True)
    for row in _rows:print(row,flush=True)
_fc_data=dict(settings=asdict(_fc_settings),control_bounds=_fc_bounds,model_run=str(_fc_run),
    lookup_error=_fc_lookup_error,outlet_rms_mismatch=_fc_cap_error,
    extension_model='Assumed fully-developed planar Poiseuille, not a new PINN/NS solution',
    particle_forces='One-way Schiller-Naumann drag only; finite-radius wall stopping; no contact/interparticle forces',
    timing='T=max(travel_age)+lead_time; release_i=T-travel_age_i; common-time spatial target',results=_fc_results)
(_fc_out/'formation_results.json').write_text(json.dumps(_fc_data,allow_nan=False))
_fc_coarse.cache.clear();_fc_verify.cache.clear()
_FC_HTML = r'''<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Optimized particle release</title>
<style>*{box-sizing:border-box}body{margin:0;background:#0b1422;color:#e9f0f8;font:14px system-ui,sans-serif}main{max-width:1350px;margin:auto;padding:20px}h2{font-size:23px;margin:0 0 8px}p{line-height:1.8;color:#b4c4d8}.bar{display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin:16px 0}button,select{font:inherit;border:1px solid #36516c;padding:9px 15px;border-radius:9px;background:#1a2a40;color:#ecf3fb;cursor:pointer}button.primary{background:#48d9b6;color:#082019;border:0}.panels{display:grid;grid-template-columns:1.7fr 1fr;gap:16px;direction:ltr}.panel{direction:rtl;background:#101e30;border:1px solid #263b54;border-radius:14px;padding:12px}canvas{width:100%;display:block}#scene{height:560px}#zoom{height:330px}.note{font-size:12px;color:#aabdce;line-height:1.8}#status{padding:14px 0;line-height:1.9}#time{flex:1;min-width:180px;direction:ltr;accent-color:#48d9b6}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font:12px monospace;direction:ltr;text-align:right}th,td{padding:10px;border-bottom:1px solid #293b51;white-space:nowrap}th{color:#a4bdd7}#clock{direction:ltr;min-width:180px;font:13px monospace}.legend{display:flex;gap:20px;justify-content:center;margin-top:8px;font-size:12px;color:#b4c4d8}.actual{color:#ffd37d}.target{color:#65e0d2}.meta{font:12px monospace;direction:ltr;white-space:pre-wrap;color:#9fb7cf}@media(max-width:850px){.panels{grid-template-columns:1fr}#scene{height:460px}#zoom{height:300px}}</style>
<main><h2>تشکیل شکل با رهایش بهینهٔ ذرات</h2><p>نقطه، دو سرعت اولیه و زمان رهایش هر ذره بهینه شده‌اند. تمام ذرات از خط ورودی می‌آیند؛ شکل در زمان مشترک T داخل امتداد خروجی تشکیل می‌شود.</p>
<div class="bar"><select id="shape" aria-label="شکل هدف"><option value="triangle">مثلث — ۳ ذره</option><option value="square">مربع — ۴ ذره</option></select><button id="play" class="primary">پخش / توقف</button><button id="again">از ابتدا</button><select id="speed" aria-label="سرعت نمایش"><option value="1">سرعت نمایش ×۱</option><option value="2">سرعت نمایش ×۲</option><option value="4">سرعت نمایش ×۴</option></select><button id="csv">دانلود برنامهٔ رهایش</button></div>
<div class="bar"><input id="time" type="range" min="0" max="1" step="0.0005" value="0" aria-label="زمان"><span id="clock"></span></div>
<div class="panels"><div class="panel"><canvas id="scene" aria-label="کانال و ذرات"></canvas><div class="note">امتداد مستطیلی: جریان سهمیِ فرض‌شده. اتصال این بخش، حل تازهٔ ناویر–استوکس یا برون‌یابی PINN نیست.</div></div><div class="panel"><canvas id="zoom" aria-label="بزرگ‌نمایی شکل هدف"></canvas><div class="legend"><span class="target">خط‌چین: هدف</span><span class="actual">نقطه: موقعیت محاسبه‌شده</span></div><div id="status"></div><div id="meta" class="meta"></div></div></div>
<div class="panel scroll" style="margin-top:16px"><table><thead><tr><th>particle</th><th>x_p0</th><th>u_p0</th><th>v_p0</th><th>release t*</th><th>travel age</th><th>target x</th><th>target y</th><th>position error</th></tr></thead><tbody id="rows"></tbody></table></div>
<p class="note">موقعیت‌ها از مسیرهای حل‌شده خوانده می‌شوند؛ ذرات برای زیباترشدن شکل به رأس‌ها جابه‌جا نمی‌شوند. تطبیق شکل، نسبت به مدل جریانِ استفاده‌شده سنجیده شده است. هدف، تشکیل شکل در یک لحظه است. همهٔ طول‌ها، سرعت‌ها و زمان‌ها بی‌بعدند.</p></main>
<script>
const DATA=__DATA__,COLORS=['#ffd37d','#86c8ff','#eea3db','#b5df89'];const $=s=>document.getElementById(s);
let current,clock=0,playing=true,last=null;const views=[$('scene'),$('zoom')];
const R=.35,H=.15,XC=1.4;function map(q,e){let sec=q>.5,a=2*Math.PI*q-(sec?Math.PI:0),s=Math.sin(a),c=Math.cos(a);if(q===0||q===.5||q===1)s=0;return sec?[3*R-R*c+H*e*c,-R*s+H*e*s]:[R-R*c-H*e*c,R*s+H*e*s];}
function point(path,age){let t=path.t,z=path.state;if(age<0)return null;if(age>=t[t.length-1])return z[z.length-1].slice(0,2);let lo=0,hi=t.length-1;while(hi-lo>1){let m=(lo+hi)>>1;if(t[m]<=age)lo=m;else hi=m;}let f=(age-t[lo])/(t[hi]-t[lo]);return[z[lo][0]*(1-f)+z[hi][0]*f,z[lo][1]*(1-f)+z[hi][1]*f];}
function transform(canvas,zoom){let w=canvas.clientWidth,h=canvas.clientHeight,extent=DATA.settings.extension;let xmin=zoom?XC-H-.02:-.22,xmax=zoom?XC+H+.02:XC+H+.15,ymin=zoom?extent/2-H-.02:-.58,ymax=zoom?extent/2+H+.02:extent+.12;let s=Math.min((w-35)/(xmax-xmin),(h-35)/(ymax-ymin));return p=>[w/2+(p[0]-(xmin+xmax)/2)*s,h/2-(p[1]-(ymin+ymax)/2)*s];}
function draw(canvas,zoom){let dpr=Math.min(2,window.devicePixelRatio||1),w=canvas.clientWidth,h=canvas.clientHeight;if(canvas.width!==Math.round(w*dpr)||canvas.height!==Math.round(h*dpr)){canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr);}let c=canvas.getContext('2d');c.setTransform(dpr,0,0,dpr,0,0);c.clearRect(0,0,w,h);let tr=transform(canvas,zoom);let line=(pts,close=false)=>{c.beginPath();pts.forEach((p,i)=>{let z=tr(p);if(i)c.lineTo(...z);else c.moveTo(...z)});if(close)c.closePath();};
let ext=[[XC-H,0],[XC+H,0],[XC+H,DATA.settings.extension],[XC-H,DATA.settings.extension]];line(ext,true);c.fillStyle='#18394a';c.fill();c.strokeStyle='#789eb2';c.lineWidth=1;line([ext[0],ext[3]]);c.stroke();line([ext[1],ext[2]]);c.stroke();
if(!zoom){let wall1=Array.from({length:301},(_,i)=>map(i/300,-1)),wall2=Array.from({length:301},(_,i)=>map(i/300,1));line(wall1.concat([...wall2].reverse()),true);c.fillStyle='#172f4b';c.fill();c.strokeStyle='#7c9bb9';line(wall1);c.stroke();line(wall2);c.stroke();c.fillStyle='#bed0e2';c.font='12px sans-serif';let p=tr([0,-.08]);c.fillText('inlet',p[0]-15,p[1]);p=tr([XC,.04]);c.fillText('extension',p[0]-27,p[1]-8);}
if(!current)return;c.setLineDash([5,5]);c.strokeStyle='#65e0d2';c.lineWidth=1.3;line(current.targets,true);c.stroke();c.setLineDash([]);for(let p of current.targets){let z=tr(p);c.strokeStyle='#65e0d2';c.beginPath();c.arc(...z,5,0,7);c.stroke();}
let actual=[];for(let i=0;i<(current.paths||[]).length;i++){let path=current.paths[i],age=clock-path.release,p=point(path,age);if(!p)continue;actual.push(p);if(!zoom){c.strokeStyle=COLORS[i]+'55';c.lineWidth=1;line(path.state.map(z=>z.slice(0,2)));c.stroke();}
let z=tr(p),zr=tr([p[0]+DATA.settings.diameter/2,p[1]]);c.fillStyle=COLORS[i];c.beginPath();c.arc(...z,Math.max(3,Math.abs(zr[0]-z[0])),0,7);c.fill();c.fillStyle='#eef6ff';c.font='12px sans-serif';c.fillText(String(i+1),z[0]+7,z[1]-7);}
if(actual.length===(current.paths||[]).length&&actual.length>0&&clock>=current.target_time-1e-7){c.strokeStyle='#ffd37d';c.lineWidth=1.5;line(actual,true);c.stroke();}}
function refresh(){for(let i=0;i<views.length;i++)draw(views[i],i===1);let T=current?.target_time||1;$('time').value=clock/T;$('clock').textContent=`t* = ${clock.toFixed(3)} / T* = ${current?.target_time?.toFixed(3)||'—'}`;}
function choose(){current=DATA.results[$('shape').value];clock=0;playing=true;last=null;$('rows').innerHTML='';for(let r of current.schedules||[]){let tr=document.createElement('tr');for(let key of ['particle','x_p0','u_p0','v_p0','release_time','age_at_target','target_x','target_y','error']){let td=document.createElement('td');td.textContent=key==='particle'?r[key]:r[key].toFixed(6);tr.appendChild(td);}$('rows').appendChild(tr);}
$('status').textContent=current.success?`هدف در مدل محقق شد | بیشینهٔ خطا: ${current.max_error.toExponential(2)} | کمترین فاصلهٔ ذرات: ${current.minimum_pair_distance.toFixed(5)}`:`در بودجهٔ جستجوی فعلی جواب مطابق معیارها پیدا نشد. ${current.reason}`;
$('meta').textContent=`Re=${DATA.settings.re}\nd_p=${DATA.settings.diameter}\nrho_p/rho_f=${DATA.settings.density}\nposition tolerance=${DATA.settings.target_tol}\nlookup relative L2=${DATA.lookup_error.toExponential(3)}\nFinal paths: direct PINN + assumed extension`;refresh();}
$('shape').onchange=choose;$('play').onclick=()=>{if(current?.target_time&&clock>=current.target_time)clock=0;playing=!playing;last=null;};$('again').onclick=()=>{clock=0;playing=true;last=null;refresh();};$('time').oninput=()=>{clock=Number($('time').value)*(current.target_time||1);playing=false;refresh();};
$('csv').onclick=()=>{let rows=current.schedules||[];if(!rows.length)return;let keys=Object.keys(rows[0]),text=keys.join(',')+'\n'+rows.map(r=>keys.map(k=>r[k]).join(',')).join('\n'),url=URL.createObjectURL(new Blob([text],{type:'text/csv'})),a=document.createElement('a');a.href=url;a.download='release_schedule_'+current.kind+'.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};
function tick(now){let T=current?.target_time||0;if(last!==null&&playing&&T>0){clock=Math.min(T,clock+Math.min(.1,(now-last)/1000)*Number($('speed').value));if(clock>=T)playing=false;}last=now;refresh();requestAnimationFrame(tick);}new ResizeObserver(refresh).observe($('scene'));new ResizeObserver(refresh).observe($('zoom'));choose();requestAnimationFrame(tick);
</script></html>
'''
_fc_page=_FC_HTML.replace('__DATA__',json.dumps(_fc_data,separators=(',',':'),allow_nan=False))
_fc_html_path=_fc_out/'formation_control.html';_fc_html_path.write_text(_fc_page,encoding='utf-8')
display(HTML('<iframe title="Optimized particle release" style="width:100%;height:1150px;border:0" sandbox="allow-scripts allow-downloads" srcdoc="'+html.escape(_fc_page,quote=True)+'"></iframe>'))
_fc_archive=shutil.make_archive(str(_fc_out),'zip',root_dir=_fc_out)
print('Download results and standalone web app:')
display(FileLink(str(_fc_html_path)))
display(FileLink(_fc_archive))


In [ ]:
# ONE NEW CELL after the completed training notebook. No network training.
from pathlib import Path
from dataclasses import asdict
import sys, importlib.util, json, csv, time, math, html, shutil
import numpy as np
import torch
from IPython.display import display, HTML, FileLink

MODEL_RUN = None  # None uses RUN from the completed notebook.
SHAPES = ['triangle','square','circle_10','s_7','s_8']
S_WIDTH, S_HEIGHT = 0.15, 0.36
CIRCLE_RADIUS = 0.10
BEAM_WIDTH = 64  # Bounded candidate search for more than four particles.
FLOW_RE = 1.0
PARTICLE_DIAMETER = 0.025
PARTICLE_DENSITY_RATIO = 1000.0
EXTENSION_LENGTH = 1.4  # 2 x centreline bend diameter (2 x 0.7), same channel width 0.3.
TARGET_RADIUS = 0.075  # Circumradius of equilateral triangle / square.
POSITION_TOLERANCE = 0.001
SEARCH_BANK_POWER = 6  # 2**6 + 11 seed launches, shared by all requested shapes.
LOCAL_STARTS = 6
SHOOTING_ITERATIONS = 12
DIRECT_PINN_ITERATIONS = 6
CHANNEL_TMAX = 20.0
EXTENSION_TMAX = 20.0
COLLISION_MARGIN = 0.001
NQ, NETA = 401, 81  # Velocity lookup for fast search; final paths use the real PINN.

_fc_sources=[Path(globals().get('ROOT','/kaggle/working/particle_corrected_source')),
             Path('/kaggle/working/particle_corrected_source'),Path('/kaggle/working/particle_injection_source')]
_fc_source=next((p for p in _fc_sources if (p/'particle_pinn/injection.py').exists()),None)
if _fc_source is None: raise FileNotFoundError('Run the setup cell of the training notebook first.')
sys.path.insert(0,str(_fc_source))
from particle_pinn.injection import load_injection_model, HALF_WIDTH, RADIUS
from particle_pinn.parametric import map_points
_fc_run=Path(MODEL_RUN) if MODEL_RUN is not None else Path(globals()['RUN']) if 'RUN' in globals() else None
if _fc_run is None or not (_fc_run/'best_model.pt').exists():
    if MODEL_RUN is not None:raise FileNotFoundError('MODEL_RUN does not contain best_model.pt')
    _fc_candidates=list(Path('/kaggle/working').glob('injection_*/best_model.pt'))
    if not _fc_candidates:raise FileNotFoundError('No best_model.pt found. Set MODEL_RUN explicitly.')
    _fc_run=max(_fc_candidates,key=lambda p:p.stat().st_mtime).parent
_fc_model,_fc_cfg=load_injection_model(_fc_run/'best_model.pt',device='cpu')
_fc_model.eval().requires_grad_(False)
torch.set_num_threads(1)
for _k,_v in [('re',FLOW_RE),('diameter',PARTICLE_DIAMETER),('density_ratio',PARTICLE_DENSITY_RATIO)]:
    if not np.isfinite(_v) or not _fc_cfg['ranges'][_k][0]<=_v<=_fc_cfg['ranges'][_k][1]:
        raise ValueError(f'{_k}={_v} is outside the saved physical range')
if min(EXTENSION_LENGTH,TARGET_RADIUS,POSITION_TOLERANCE,CHANNEL_TMAX,EXTENSION_TMAX)<=0 or not np.isfinite([EXTENSION_LENGTH,TARGET_RADIUS,POSITION_TOLERANCE,CHANNEL_TMAX,EXTENSION_TMAX,COLLISION_MARGIN]).all() or COLLISION_MARGIN<0:
    raise ValueError('Invalid formation settings')
if not 0<=SEARCH_BANK_POWER<=10 or not 1<=LOCAL_STARTS<=8 or min(SHOOTING_ITERATIONS,DIRECT_PINN_ITERATIONS)<1 or min(NQ,NETA)<3:
    raise ValueError('Invalid optimization or lookup budget')
_fc_clear=HALF_WIDTH-PARTICLE_DIAMETER/2-1e-6
_fc_bounds=[[max(_fc_cfg['ranges']['x_p0'][0],-_fc_clear),min(_fc_cfg['ranges']['x_p0'][1],_fc_clear)],
            list(_fc_cfg['ranges']['u_p0']),list(_fc_cfg['ranges']['v_p0'])]
if _fc_bounds[0][0]>_fc_bounds[0][1]:raise ValueError('No admissible inlet position for this particle')
_fc_out=_fc_run/('formation_control_'+str(time.time_ns()));_fc_out.mkdir()
print('Frozen model:',_fc_run/'best_model.pt',flush=True)
print('Output:',_fc_out,flush=True)

# Export the physical search engine beside the results, without altering model source/weights.
_FC_CORE = r'''"""Inverse release control in a frozen steady flow; no neural training."""
from dataclasses import dataclass
from itertools import product
import math,time
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar
from scipy.stats import qmc

R=.35; H=.15; LENGTH=2*math.pi*R; XC=4*R

def coordinates(x,y):
    second=x>3*R or (x>=R and y<0)
    dx=x-(3*R if second else R);dy=-y if second else y
    a=math.pi-math.atan2(max(dy,0.),dx)
    s=(a+math.pi if second else a)*R
    rad=math.hypot(dx,y);eta=(R-rad if second else rad-R)/H
    if x<R and y<0:s=y;eta=-x/H
    if x>3*R and y>0:s=LENGTH+y;eta=(x-XC)/H
    return s,eta

def map_point(q,e):
    second=q>.5;a=2*math.pi*q-(math.pi if second else 0.)
    si=math.sin(a);co=math.cos(a)
    if q in (0.,.5,1.):si=0.
    return (3*R-R*co+H*e*co,-R*si+H*e*si) if second else (R-R*co-H*e*co,R*si+H*e*si)

@dataclass
class Settings:
    re:float=1.
    diameter:float=.025
    density:float=1000.
    extension:float=1.4
    target_radius:float=.075
    s_width:float=.15
    s_height:float=.36
    circle_radius:float=.10
    beam_width:int=64
    target_tol:float=.001
    channel_tmax:float=20.
    extension_tmax:float=20.
    bank_power:int=6
    starts:int=3
    iterations:int=12
    direct_iterations:int=6
    collision_margin:float=.001
    lead_time:float=.5
    seed:int=42

class Trajectory:
    def __init__(self,control,channel,extension,status):
        self.control=np.array(control,float);self.channel=channel;self.extension=extension;self.status=status
    def hit(self,target_y):
        if self.extension is None:return None
        sol=self.extension
        # First upward crossing of the desired y, using dense output and root finding.
        crossings=np.flatnonzero((sol.y[1,:-1]<=target_y)&(sol.y[1,1:]>=target_y))
        if not len(crossings):return None
        i=crossings[0]
        tau=brentq(lambda t:sol.sol(t)[1]-target_y,sol.t[i],sol.t[i+1],xtol=1e-11)
        state=sol.sol(tau)
        return dict(age=float(self.channel.t[-1]+tau),state=state)
    def state(self,ages):
        ages=np.atleast_1d(ages);out=np.empty((len(ages),4));tc=self.channel.t[-1]
        before=ages<=tc
        if np.any(before):out[before]=self.channel.sol(np.clip(ages[before],0,tc)).T
        if np.any(~before):out[~before]=self.extension.sol(ages[~before]-tc).T
        return out

class Planner:
    def __init__(self,field,settings,bounds,tag='lookup'):
        self.field=field;self.c=settings;self.bounds=np.array(bounds,float);self.tag=tag;self.cache={};self.evaluations=0
        if self.bounds.shape!=(3,2) or np.any(self.bounds[:,0]>self.bounds[:,1]):raise ValueError('Invalid control bounds')
        self.span=self.bounds[:,1]-self.bounds[:,0]
    def physical(self,z):return self.bounds[:,0]+np.clip(z,0,1)*self.span
    def normalized(self,u):return np.divide(np.asarray(u)-self.bounds[:,0],self.span,out=np.zeros(3),where=self.span>0)
    def _acceleration(self,fluid,z):
        du,dv=fluid[0]-z[2],fluid[1]-z[3]
        rep=self.c.re*self.c.diameter*math.hypot(du,dv)
        f=1+.15*rep**.687 if rep<1000 else .44*rep/24
        k=18*f/(self.c.density*self.c.re*self.c.diameter**2)
        return du*k,dv*k
    def path(self,z):
        control=self.physical(z);key=tuple(np.round(control,11))
        if key in self.cache:return self.cache[key]
        self.evaluations+=1;c=self.c;x0,u0,v0=control;evals=0
        radius=c.diameter/2
        if abs(x0)+radius>=H or v0<0:
            result=Trajectory(control,None,None,'invalid_injection');self.cache[key]=result;return result
        def rhs(t,state):
            nonlocal evals
            evals+=1
            if evals>30000:raise RuntimeError('Channel RHS budget exceeded')
            sx,et=coordinates(state[0],state[1])
            # Only event-bracketing stages may leave the domain: query a boundary
            # projection there, never interpret PINN extrapolation as an extension.
            fx,fy=map_point(min(1.,max(0.,sx/LENGTH)),min(1.,max(-1.,et))) if (sx<0 or sx>LENGTH or abs(et)>1) else (state[0],state[1])
            fluid=self.field(fx,fy)
            if not np.isfinite(fluid).all():raise FloatingPointError('Non-finite velocity')
            ax,ay=self._acceleration(fluid,state)
            return [state[2],state[3],ax,ay]
        def wall(t,z):return H*(1-abs(coordinates(z[0],z[1])[1]))-radius
        def outlet(t,z):return LENGTH-coordinates(z[0],z[1])[0]
        def inlet(t,z):return 1e-12 if t==0 and v0>=0 else coordinates(z[0],z[1])[0]
        for ev in [wall,outlet,inlet]:ev.terminal=True;ev.direction=-1
        tau=c.density*c.re*c.diameter**2/18
        method='Radau' if tau<.005 else 'RK45'
        step=min(.025,H/(10*max(1,math.hypot(u0,v0))))
        try:
            ch=solve_ivp(rhs,(0,c.channel_tmax),[x0,0,u0,v0],events=[wall,outlet,inlet],
                dense_output=True,rtol=2e-6,atol=2e-8,max_step=step,method=method)
            if not ch.success:status='channel_solver_failed'
            elif len(ch.t_events[0]):status='channel_wall'
            elif len(ch.t_events[2]):status='inlet_exit'
            elif not len(ch.t_events[1]):status='channel_time_limit'
            else:status='outlet'
            ex=None
            if status=='outlet':
                start=ch.y[:,-1].copy();start[1]=0. # exact cap; position/velocity otherwise preserved
                if abs(start[0]-XC)+radius>=H:status='extension_initial_wall'
                else:
                    evals=0
                    def erhs(t,z):
                        nonlocal evals
                        evals+=1
                        if evals>30000:raise RuntimeError('Extension RHS budget exceeded')
                        fluid=(0.,max(0.,1-((z[0]-XC)/H)**2))
                        ax,ay=self._acceleration(fluid,z)
                        return [z[2],z[3],ax,ay]
                    def ewall(t,z):return H-abs(z[0]-XC)-radius
                    def etop(t,z):return c.extension-z[1]
                    def eback(t,z):return 1e-12 if t==0 and start[3]>=0 else z[1]
                    for ev in [ewall,etop,eback]:ev.terminal=True;ev.direction=-1
                    ex=solve_ivp(erhs,(0,c.extension_tmax),start,events=[ewall,etop,eback],
                        dense_output=True,rtol=2e-7,atol=2e-9,max_step=step,method=method)
                    status='extension_solver_failed' if not ex.success else 'extension_wall' if len(ex.t_events[0]) else 'extension_end' if len(ex.t_events[1]) else 'extension_backflow' if len(ex.t_events[2]) else 'extension_time_limit'
            result=Trajectory(control,ch,ex,status)
        except (RuntimeError,ValueError,FloatingPointError) as error:
            result=Trajectory(control,None,None,'failed: '+str(error))
        self.cache[key]=result
        return result
    def bank(self):
        samples=qmc.Sobol(d=3,scramble=True,seed=self.c.seed).random_base2(self.c.bank_power)
        base=np.array([0.,0.,1.]);base=np.clip(base,self.bounds[:,0],self.bounds[:,1])
        anchors=np.tile(self.normalized(base),(11,1));anchors[:,0]=np.linspace(.001,.999,11)
        samples=np.vstack((anchors,samples))
        bank=[];started=time.monotonic()
        for i,z in enumerate(samples):
            p=self.path(z)
            if p.extension is not None and p.extension.success:bank.append((z,p))
            if (i+1)%16==0:print(f'{self.tag}: search {i+1}/{len(samples)}, exit paths {len(bank)}, {time.monotonic()-started:.1f}s',flush=True)
        return bank
    def refine(self,z,target,iterations=None):
        z=np.clip(np.asarray(z,float),0,1);best=None;trace=[]
        for it in range(self.c.iterations if iterations is None else iterations):
            path=self.path(z);hit=path.hit(target[1])
            if hit is None:break
            error=float(hit['state'][0]-target[0])
            if best is None or abs(error)<best['error']:
                best=dict(z=z.copy(),path=path,hit=hit,error=abs(error))
            trace.append(abs(error))
            if abs(error)<=self.c.target_tol*.15:break
            # Absolute normalized perturbations avoid noisy near-zero relative differences.
            jac=np.zeros(3);delta=.002
            for j in range(3):
                if self.span[j]==0:continue
                zp=z.copy();zm=z.copy();zp[j]=min(1,z[j]+delta);zm[j]=max(0,z[j]-delta)
                hp=self.path(zp).hit(target[1]);hm=self.path(zm).hit(target[1])
                if hp is not None and hm is not None:
                    jac[j]=(hp['state'][0]-hm['state'][0])/(zp[j]-zm[j])
                elif hp is not None and zp[j]>z[j]:jac[j]=(hp['state'][0]-hit['state'][0])/(zp[j]-z[j])
                elif hm is not None and zm[j]<z[j]:jac[j]=(hit['state'][0]-hm['state'][0])/(z[j]-zm[j])
            norm=float(jac@jac)
            if norm<1e-14:break
            move=-error*jac/(norm+1e-10)
            move*=min(1.,.18/max(np.max(np.abs(move)),1e-15))
            improved=False
            for factor in [1.,.5,.25,.125,.0625]:
                trial=np.clip(z+factor*move,0,1);h=self.path(trial).hit(target[1])
                if h is not None and abs(h['state'][0]-target[0])<abs(error):
                    z=trial;improved=True;break
            if not improved:break
        last_path=self.path(z);last_hit=last_path.hit(target[1])
        if last_hit is not None:
            error=abs(float(last_hit['state'][0]-target[0]))
            if best is None or error<best['error']:best=dict(z=z.copy(),path=last_path,hit=last_hit,error=error)
        if best is not None:best['optimization_errors']=trace
        return best


def targets(kind,c):
    if kind in ('triangle','square'):
        n=3 if kind=='triangle' else 4
        phase=math.pi/2 if n==3 else math.pi/4
        angle=phase+np.arange(n)*2*math.pi/n
        pts=np.column_stack((XC+c.target_radius*np.cos(angle),c.extension/2+c.target_radius*np.sin(angle)))
    elif kind.startswith('circle_'):
        n=int(kind.split('_')[1])
        if not 3<=n<=40:raise ValueError('Circle particle count must be 3..40')
        angle=math.pi/2+np.arange(n)*2*math.pi/n
        pts=np.column_stack((XC+c.circle_radius*np.cos(angle),c.extension/2+c.circle_radius*np.sin(angle)))
    elif kind.startswith('s_'):
        n=int(kind.split('_')[1])
        if not 5<=n<=40:raise ValueError('S particle count must be 5..40')
        if not np.isfinite([c.s_width,c.s_height]).all() or min(c.s_width,c.s_height)<=0:raise ValueError('Positive S width and height required')
        t=np.linspace(0,1,1001)[:,None]
        def bezier(a,b,d,e):return (1-t)**3*a+3*(1-t)**2*t*b+3*(1-t)*t*t*d+t**3*e
        curve=np.vstack((bezier(np.array([.75,.85]),np.array([-1.4,1.4]),np.array([-1.5,.1]),np.array([0.,0.])),
                         bezier(np.array([0.,0.]),np.array([1.5,-.1]),np.array([1.4,-1.4]),np.array([-.75,-.85]))[1:]))
        curve[:,0]*=c.s_width/np.ptp(curve[:,0]);curve[:,1]*=c.s_height/np.ptp(curve[:,1])
        distance=np.r_[0.,np.cumsum(np.linalg.norm(np.diff(curve,axis=0),axis=1))]
        knots=np.linspace(0,distance[-1],n)
        pts=np.column_stack((np.interp(knots,distance,curve[:,0])+XC,np.interp(knots,distance,curve[:,1])+c.extension/2))
    else:raise ValueError('Use triangle, square, circle_N or s_N')
    if np.any(abs(pts[:,0]-XC)+c.diameter/2>=H) or np.any(pts[:,1]-c.diameter/2<=0) or np.any(pts[:,1]+c.diameter/2>=c.extension):
        raise ValueError('Target does not fit the extension with particle-radius clearance')
    distances=np.linalg.norm(pts[:,None,:]-pts[None,:,:],axis=2)+np.eye(n)*1000
    if distances.min()<c.diameter+c.collision_margin:raise ValueError('Target vertices are too close for the particle size')
    return pts


def separation(candidates,c,refine=False):
    ages=np.array([a['hit']['age'] for a in candidates]);T=float(ages.max()+c.lead_time);release=T-ages
    minimum=float('inf')
    for i in range(len(candidates)):
        for j in range(i+1,len(candidates)):
            low=max(release[i],release[j]);grid=np.linspace(low,T,161)
            def distance(t):
                a=candidates[i]['path'].state(np.atleast_1d(t)-release[i])[:,:2]
                b=candidates[j]['path'].state(np.atleast_1d(t)-release[j])[:,:2]
                return np.linalg.norm(a-b,axis=1)
            d=distance(grid);minimum=min(minimum,float(d.min()))
            if refine:
                minima=np.flatnonzero((d[1:-1]<=d[:-2])&(d[1:-1]<=d[2:]))+1
                for k in minima:
                    r=minimize_scalar(lambda t:float(distance(t)[0]),bounds=(grid[k-1],grid[k+1]),method='bounded',options={'xatol':1e-8})
                    minimum=min(minimum,float(r.fun))
    return T,release,minimum


def plan_shape(kind,coarse,direct,bank):
    c=coarse.c;goal=targets(kind,c);options=[];diagnostics=[]
    for index,target in enumerate(goal):
        pool=[]
        for z,p in bank:
            hit=p.hit(target[1])
            if hit is not None:pool.append((abs(hit['state'][0]-target[0]),z))
        pool.sort(key=lambda item:item[0])
        if not pool:
            return dict(kind=kind,success=False,reason=f'No feasible search trajectory reaches target height for vertex {index+1}',targets=goal.tolist(),details=diagnostics)
        candidates=[]
        # Preserve diverse speed/position seeds, rather than several almost identical paths.
        remaining=pool[:max(48,8*c.starts)];seeds=[remaining.pop(0)[1]]
        while remaining and len(seeds)<c.starts:
            k=max(range(len(remaining)),key=lambda k:min(np.linalg.norm(remaining[k][1]-z) for z in seeds)-.15*remaining[k][0]/H)
            seeds.append(remaining.pop(k)[1])
        for z in seeds:
            best=coarse.refine(z,target)
            if best is None:continue
            # Verify/refine using the real frozen PINN, not the lookup table.
            verified=direct.refine(best['z'],target,c.direct_iterations)
            if verified is not None and all(np.linalg.norm(verified['z']-p['z'])>1e-5 for p in candidates):candidates.append(verified)
        candidates.sort(key=lambda b:b['error'])
        diagnostics.append(dict(vertex=index+1,available_seeds=len(pool),verified_candidates=len(candidates)))
        print(f'{kind} vertex {index+1}: '+('best direct error %.6g'%candidates[0]['error'] if candidates else 'no verified path'),flush=True)
        if not candidates:
            return dict(kind=kind,success=False,reason=f'No direct-PINN verified path for vertex {index+1}',targets=goal.tolist(),details=diagnostics)
        options.append(candidates[:c.starts])
    ranked=[]
    if len(options)<=4:
        for combo in product(*options):
            T,release,sep=separation(combo,c)
            error=max(p['error'] for p in combo)
            key=(error>c.target_tol,sep<c.diameter+c.collision_margin,error,T)
            ranked.append((key,combo))
    else:
        # Relative release times depend only on each pair's travel ages.
        # Consequently pair clearances can be cached independently of other particles.
        pair_cache={}
        beam=[((),(),float('inf'),0.,0.)]
        for i,pool in enumerate(options):
            expanded=[]
            for indices,combo,minimum,maxerror,maxage in beam:
                for k,item in enumerate(pool):
                    sep=minimum
                    for j,old in enumerate(combo):
                        key=(j,indices[j],i,k)
                        if key not in pair_cache:pair_cache[key]=separation([old,item],c)[2]
                        sep=min(sep,pair_cache[key])
                    err=max(maxerror,item['error']);age=max(maxage,item['hit']['age'])
                    new=(indices+(k,),combo+(item,),sep,err,age)
                    score=(err>c.target_tol,sep<c.diameter+c.collision_margin,
                           max(0.,c.diameter+c.collision_margin-sep),err,age)
                    expanded.append((score,new))
            expanded.sort(key=lambda entry:entry[0]);beam=[entry[1] for entry in expanded[:c.beam_width]]
        for indices,combo,sep,error,age in beam:
            key=(error>c.target_tol,sep<c.diameter+c.collision_margin,error,age+c.lead_time)
            ranked.append((key,combo))
    ranked.sort(key=lambda item:item[0])
    chosen=None;score=None
    # Check alternatives if a narrow close approach was missed by coarse sampling.
    possible=[]
    for key,candidate in ranked:
        # A sampled distance already below clearance proves this combination invalid.
        if not key[0] and not key[1]:possible.append((key,candidate))
    if not possible:possible=ranked[:1]
    for _,candidate in possible:
        T,release,sep=separation(candidate,c,refine=True)
        error=max(p['error'] for p in candidate)
        key=(error>c.target_tol,sep<c.diameter+c.collision_margin,error,T)
        if score is None or key<score:score=key;chosen=(candidate,T,release,sep)
        if error<=c.target_tol and sep>=c.diameter+c.collision_margin:break
    combo,T,release,sep=chosen
    actual=np.array([p['hit']['state'][:2] for p in combo]);errors=np.linalg.norm(actual-goal,axis=1)
    schedules=[];paths=[]
    for i,p in enumerate(combo):
        xp,up,vp=p['path'].control;age=p['hit']['age']
        schedules.append(dict(particle=i+1,x_p0=float(xp),u_p0=float(up),v_p0=float(vp),release_time=float(release[i]),
            age_at_target=float(age),target_x=float(goal[i,0]),target_y=float(goal[i,1]),actual_x=float(actual[i,0]),actual_y=float(actual[i,1]),error=float(errors[i])))
        ts=np.linspace(0,age,700);state=p['path'].state(ts)
        paths.append(dict(t=ts.tolist(),state=state.tolist(),release=float(release[i])))
    success=bool(errors.max()<=c.target_tol and sep>=c.diameter+c.collision_margin)
    return dict(kind=kind,position_success=bool(errors.max()<=c.target_tol),clearance_success=bool(sep>=c.diameter+c.collision_margin),closed=not kind.startswith('s_'),selection='exhaustive' if len(goal)<=4 else 'bounded_beam',beam_width=c.beam_width,success=success,reason='target_and_clearance_met' if success else 'best_found_solution_misses_target_or_clearance',
        target_time=T,targets=goal.tolist(),actual=actual.tolist(),max_error=float(errors.max()),rms_error=float(np.sqrt(np.mean(errors**2))),
        minimum_pair_distance=sep,required_pair_distance=c.diameter+c.collision_margin,
        schedules=schedules,paths=paths,details=diagnostics,
        collision_check='Dense time sampling plus bounded refinement of observed local distance minima; no particle contact forces modeled')
'''
(_fc_out/'formation_core.py').write_text(_FC_CORE,encoding='utf-8')
_fc_spec=importlib.util.spec_from_file_location('particle_formation_core',_fc_out/'formation_core.py')
_fc=importlib.util.module_from_spec(_fc_spec);sys.modules[_fc_spec.name]=_fc;_fc_spec.loader.exec_module(_fc)

_fc_settings=_fc.Settings(re=float(FLOW_RE),diameter=float(PARTICLE_DIAMETER),density=float(PARTICLE_DENSITY_RATIO),
    extension=float(EXTENSION_LENGTH),target_radius=float(TARGET_RADIUS),target_tol=float(POSITION_TOLERANCE),
    channel_tmax=float(CHANNEL_TMAX),extension_tmax=float(EXTENSION_TMAX),bank_power=int(SEARCH_BANK_POWER),
    starts=int(LOCAL_STARTS),iterations=int(SHOOTING_ITERATIONS),direct_iterations=int(DIRECT_PINN_ITERATIONS),
    s_width=float(S_WIDTH),s_height=float(S_HEIGHT),circle_radius=float(CIRCLE_RADIUS),beam_width=int(BEAM_WIDTH),
    collision_margin=float(COLLISION_MARGIN),seed=int(_fc_cfg.get('seed',42)))
if not SHAPES or not all(isinstance(name,str) for name in SHAPES) or len(SHAPES)>12 or len(set(SHAPES))!=len(SHAPES):raise ValueError('Use 1..12 unique shape names')
if not 1<=BEAM_WIDTH<=256:raise ValueError('BEAM_WIDTH must be 1..256')

def _fc_velocity_batch(points):
    with torch.no_grad():
        # For v2 the actual velocity is the raw two-component output; avoid the
        # unnecessary second pressure-gauge pass on each scalar ODE query.
        if _fc_cfg.get('flow_variant')=='smooth_cartesian_v2':
            x=torch.as_tensor(np.atleast_2d(points),dtype=torch.float32)
            rr=x.new_full((len(x),1),FLOW_RE)
            p=_fc_model.geometry_parameters(rr)
            return _fc_model.net(_fc_model.features(x,p))[:,:2].cpu().numpy()
        return _fc_model.evaluate(np.atleast_2d(points),FLOW_RE)[:,:2]

def _fc_direct(x,y):return _fc_velocity_batch([[x,y]])[0]

_q,_e=torch.meshgrid(torch.linspace(0,1,NQ),torch.linspace(-1,1,NETA),indexing='ij')
_pp=_fc_model.geometry_parameters(torch.full((_q.numel(),1),FLOW_RE))
_xy=map_points(_q.flatten(),_e.flatten(),_pp).numpy()
_fc_uv=np.concatenate([_fc_velocity_batch(_xy[k:k+2048]) for k in range(0,len(_xy),2048)]).reshape(NQ,NETA,2)
if not np.isfinite(_fc_uv).all():raise FloatingPointError('Non-finite PINN velocity')
def _fc_lookup(x,y):
    s,e=_fc.coordinates(x,y)
    a=min(1.,max(0.,s/_fc.LENGTH))*(NQ-1);b=(min(1.,max(-1.,e))+1)*.5*(NETA-1)
    i=min(int(a),NQ-2);j=min(int(b),NETA-2);f=a-i;g=b-j
    return (1-f)*(1-g)*_fc_uv[i,j]+f*(1-g)*_fc_uv[i+1,j]+(1-f)*g*_fc_uv[i,j+1]+f*g*_fc_uv[i+1,j+1]

_frng=np.random.default_rng(789)
_qq=_frng.uniform(0,1,1200);_ee=_frng.uniform(-.99,.99,1200)
_testxy=np.array([_fc.map_point(float(q),float(e)) for q,e in zip(_qq,_ee)])
_truth=_fc_velocity_batch(_testxy);_approx=np.array([_fc_lookup(float(x),float(y)) for x,y in _testxy])
_fc_lookup_error=float(np.linalg.norm(_approx-_truth)/max(np.linalg.norm(_truth),1e-12))
# Compare the trained cap velocity with the nominal analytic profile used in the extension.
_xcap=np.linspace(_fc.XC-HALF_WIDTH,_fc.XC+HALF_WIDTH,101)
_cap=_fc_velocity_batch(np.column_stack((_xcap,np.zeros_like(_xcap))))
_nominal=np.column_stack((np.zeros_like(_xcap),1-((_xcap-_fc.XC)/HALF_WIDTH)**2))
_fc_cap_error=float(np.sqrt(np.mean((_cap-_nominal)**2)))
print(f'Lookup velocity relative L2 vs PINN: {_fc_lookup_error:.3%}',flush=True)
print(f'Outlet PINN / assumed-extension velocity RMS mismatch: {_fc_cap_error:.6g}',flush=True)
print('Extension assumption: u=0, v=1-((x-1.4)/0.15)^2; no PINN extrapolation or new CFD solve.',flush=True)
_fc_coarse=_fc.Planner(_fc_lookup,_fc_settings,_fc_bounds,'fast search')
_fc_verify=_fc.Planner(_fc_direct,_fc_settings,_fc_bounds,'direct PINN verification')
_fc_bank=_fc_coarse.bank()
_fc_results={}
for _shape in SHAPES:
    _start=time.monotonic()
    try:_result=_fc.plan_shape(_shape,_fc_coarse,_fc_verify,_fc_bank)
    except Exception as error:
        import traceback
        _result=dict(kind=_shape,success=False,reason=str(error),traceback=traceback.format_exc(),targets=[],closed=not _shape.startswith('s_'))
    _result['optimization_seconds']=time.monotonic()-_start
    _fc_results[_shape]=_result
    (_fc_out/(_shape+'_result.json')).write_text(json.dumps(_result,indent=2,allow_nan=False))
    _rows=_result.get('schedules',[])
    if _rows:
        with (_fc_out/(_shape+'_release_schedule.csv')).open('w',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=list(_rows[0]));writer.writeheader();writer.writerows(_rows)
        for i,p in enumerate(_result['paths']):
            arr=np.column_stack((p['t'],np.array(p['t'])+p['release'],p['state']))
            np.savetxt(_fc_out/f'{_shape}_particle_{i+1}.csv',arr,delimiter=',',comments='',header='age,global_time,x,y,u_p,v_p')
    print(_shape, '| success:',_result['success'],'| max error:',_result.get('max_error'),
          '| minimum pair distance:',_result.get('minimum_pair_distance'),flush=True)
    for row in _rows:print(row,flush=True)
_fc_data=dict(settings=asdict(_fc_settings),control_bounds=_fc_bounds,model_run=str(_fc_run),
    lookup_error=_fc_lookup_error,outlet_rms_mismatch=_fc_cap_error,
    extension_model='Assumed fully-developed planar Poiseuille, not a new PINN/NS solution',
    particle_forces='One-way Schiller-Naumann drag only; finite-radius wall stopping; no contact/interparticle forces',
    timing='T=max(travel_age)+lead_time; release_i=T-travel_age_i; common-time spatial target',results=_fc_results)
(_fc_out/'formation_results.json').write_text(json.dumps(_fc_data,allow_nan=False))
_fc_coarse.cache.clear();_fc_verify.cache.clear()
_FC_HTML = r'''<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Optimized particle release</title>
<style>*{box-sizing:border-box}body{margin:0;background:#0b1422;color:#e9f0f8;font:14px system-ui,sans-serif}main{max-width:1350px;margin:auto;padding:20px}h2{font-size:23px;margin:0 0 8px}p{line-height:1.8;color:#b4c4d8}.bar{display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin:16px 0}button,select{font:inherit;border:1px solid #36516c;padding:9px 15px;border-radius:9px;background:#1a2a40;color:#ecf3fb;cursor:pointer}button.primary{background:#48d9b6;color:#082019;border:0}.panels{display:grid;grid-template-columns:1.7fr 1fr;gap:16px;direction:ltr}.panel{direction:rtl;background:#101e30;border:1px solid #263b54;border-radius:14px;padding:12px}canvas{width:100%;display:block}#scene{height:560px}#zoom{height:330px}.note{font-size:12px;color:#aabdce;line-height:1.8}#status{padding:14px 0;line-height:1.9}#time{flex:1;min-width:180px;direction:ltr;accent-color:#48d9b6}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font:12px monospace;direction:ltr;text-align:right}th,td{padding:10px;border-bottom:1px solid #293b51;white-space:nowrap}th{color:#a4bdd7}#clock{direction:ltr;min-width:180px;font:13px monospace}.legend{display:flex;gap:20px;justify-content:center;margin-top:8px;font-size:12px;color:#b4c4d8}.actual{color:#ffd37d}.target{color:#65e0d2}.meta{font:12px monospace;direction:ltr;white-space:pre-wrap;color:#9fb7cf}@media(max-width:850px){.panels{grid-template-columns:1fr}#scene{height:460px}#zoom{height:300px}}</style>
<main><h2>تشکیل شکل با رهایش بهینهٔ ذرات</h2><p>نقطه، دو سرعت اولیه و زمان رهایش هر ذره بهینه شده‌اند. تمام ذرات از خط ورودی می‌آیند؛ شکل در زمان مشترک T داخل امتداد خروجی تشکیل می‌شود.</p>
<div class="bar"><select id="shape" aria-label="شکل هدف"><option value="triangle">مثلث — ۳ ذره</option><option value="square">مربع — ۴ ذره</option></select><button id="play" class="primary">پخش / توقف</button><button id="again">از ابتدا</button><select id="speed" aria-label="سرعت نمایش"><option value="1">سرعت نمایش ×۱</option><option value="2">سرعت نمایش ×۲</option><option value="4">سرعت نمایش ×۴</option></select><button id="csv">دانلود برنامهٔ رهایش</button></div>
<div class="bar"><input id="time" type="range" min="0" max="1" step="0.0005" value="0" aria-label="زمان"><span id="clock"></span></div>
<div class="panels"><div class="panel"><canvas id="scene" aria-label="کانال و ذرات"></canvas><div class="note">امتداد مستطیلی: جریان سهمیِ فرض‌شده. اتصال این بخش، حل تازهٔ ناویر–استوکس یا برون‌یابی PINN نیست.</div></div><div class="panel"><canvas id="zoom" aria-label="بزرگ‌نمایی شکل هدف"></canvas><div class="legend"><span class="target">خط‌چین: هدف</span><span class="actual">نقطه: موقعیت محاسبه‌شده</span></div><div id="status"></div><div id="meta" class="meta"></div></div></div>
<div class="panel scroll" style="margin-top:16px"><table><thead><tr><th>particle</th><th>x_p0</th><th>u_p0</th><th>v_p0</th><th>release t*</th><th>travel age</th><th>target x</th><th>target y</th><th>position error</th></tr></thead><tbody id="rows"></tbody></table></div>
<p class="note">موقعیت‌ها از مسیرهای حل‌شده خوانده می‌شوند؛ ذرات برای زیباترشدن شکل به رأس‌ها جابه‌جا نمی‌شوند. تطبیق شکل، نسبت به مدل جریانِ استفاده‌شده سنجیده شده است. هدف، تشکیل شکل در یک لحظه است. همهٔ طول‌ها، سرعت‌ها و زمان‌ها بی‌بعدند.</p></main>
<script>
const DATA=__DATA__,COLORS=['#ffd37d','#86c8ff','#eea3db','#b5df89','#f69583','#a9a2ff','#78d5c2','#e1c9a8','#8ddce8','#dda1b3','#b8d0f5','#d2d68c'];const $=s=>document.getElementById(s);
let current,clock=0,playing=true,last=null;const views=[$('scene'),$('zoom')];
const R=.35,H=.15,XC=1.4;function map(q,e){let sec=q>.5,a=2*Math.PI*q-(sec?Math.PI:0),s=Math.sin(a),c=Math.cos(a);if(q===0||q===.5||q===1)s=0;return sec?[3*R-R*c+H*e*c,-R*s+H*e*s]:[R-R*c-H*e*c,R*s+H*e*s];}
function point(path,age){let t=path.t,z=path.state;if(age<0)return null;if(age>=t[t.length-1])return z[z.length-1].slice(0,2);let lo=0,hi=t.length-1;while(hi-lo>1){let m=(lo+hi)>>1;if(t[m]<=age)lo=m;else hi=m;}let f=(age-t[lo])/(t[hi]-t[lo]);return[z[lo][0]*(1-f)+z[hi][0]*f,z[lo][1]*(1-f)+z[hi][1]*f];}
function transform(canvas,zoom){let w=canvas.clientWidth,h=canvas.clientHeight,extent=DATA.settings.extension;let zh=Math.max(H+.02,(DATA.settings.s_height||.24)/2+.025);let xmin=zoom?XC-H-.02:-.22,xmax=zoom?XC+H+.02:XC+H+.15,ymin=zoom?extent/2-zh:-.58,ymax=zoom?extent/2+zh:extent+.12;let s=Math.min((w-35)/(xmax-xmin),(h-35)/(ymax-ymin));return p=>[w/2+(p[0]-(xmin+xmax)/2)*s,h/2-(p[1]-(ymin+ymax)/2)*s];}
function draw(canvas,zoom){let dpr=Math.min(2,window.devicePixelRatio||1),w=canvas.clientWidth,h=canvas.clientHeight;if(canvas.width!==Math.round(w*dpr)||canvas.height!==Math.round(h*dpr)){canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr);}let c=canvas.getContext('2d');c.setTransform(dpr,0,0,dpr,0,0);c.clearRect(0,0,w,h);let tr=transform(canvas,zoom);let line=(pts,close=false)=>{c.beginPath();pts.forEach((p,i)=>{let z=tr(p);if(i)c.lineTo(...z);else c.moveTo(...z)});if(close)c.closePath();};
let ext=[[XC-H,0],[XC+H,0],[XC+H,DATA.settings.extension],[XC-H,DATA.settings.extension]];line(ext,true);c.fillStyle='#18394a';c.fill();c.strokeStyle='#789eb2';c.lineWidth=1;line([ext[0],ext[3]]);c.stroke();line([ext[1],ext[2]]);c.stroke();
if(!zoom){let wall1=Array.from({length:301},(_,i)=>map(i/300,-1)),wall2=Array.from({length:301},(_,i)=>map(i/300,1));line(wall1.concat([...wall2].reverse()),true);c.fillStyle='#172f4b';c.fill();c.strokeStyle='#7c9bb9';line(wall1);c.stroke();line(wall2);c.stroke();c.fillStyle='#bed0e2';c.font='12px sans-serif';let p=tr([0,-.08]);c.fillText('inlet',p[0]-15,p[1]);p=tr([XC,.04]);c.fillText('extension',p[0]-27,p[1]-8);}
if(!current)return;c.setLineDash([5,5]);c.strokeStyle='#65e0d2';c.lineWidth=1.3;line(current.targets,current.closed!==false);c.stroke();c.setLineDash([]);for(let p of current.targets){let z=tr(p);c.strokeStyle='#65e0d2';c.beginPath();c.arc(...z,5,0,7);c.stroke();}
let actual=[];for(let i=0;i<(current.paths||[]).length;i++){let path=current.paths[i],age=clock-path.release,p=point(path,age);if(!p)continue;actual.push(p);if(!zoom){c.strokeStyle=COLORS[i%COLORS.length]+'55';c.lineWidth=1;line(path.state.map(z=>z.slice(0,2)));c.stroke();}
let z=tr(p),zr=tr([p[0]+DATA.settings.diameter/2,p[1]]);c.fillStyle=COLORS[i%COLORS.length];c.beginPath();c.arc(...z,Math.max(3,Math.abs(zr[0]-z[0])),0,7);c.fill();c.fillStyle='#eef6ff';c.font='12px sans-serif';c.fillText(String(i+1),z[0]+7,z[1]-7);}
if(actual.length===(current.paths||[]).length&&actual.length>0&&clock>=current.target_time-1e-7){c.strokeStyle='#ffd37d';c.lineWidth=1.5;line(actual,current.closed!==false);c.stroke();}}
function refresh(){for(let i=0;i<views.length;i++)draw(views[i],i===1);let T=current?.target_time||1;$('time').value=clock/T;$('clock').textContent=`t* = ${clock.toFixed(3)} / T* = ${current?.target_time?.toFixed(3)||'—'}`;}
function choose(){current=DATA.results[$('shape').value];clock=0;playing=true;last=null;$('rows').innerHTML='';for(let r of current.schedules||[]){let tr=document.createElement('tr');for(let key of ['particle','x_p0','u_p0','v_p0','release_time','age_at_target','target_x','target_y','error']){let td=document.createElement('td');td.textContent=key==='particle'?r[key]:r[key].toFixed(6);tr.appendChild(td);}$('rows').appendChild(tr);}
$('status').textContent=current.success?`هدف در مدل محقق شد | بیشینهٔ خطا: ${current.max_error.toExponential(2)} | کمترین فاصلهٔ ذرات: ${current.minimum_pair_distance.toFixed(5)}`:current.position_success?`نقاط هدف با دقت مجاز حاصل شدند، اما کنترل فاصلهٔ ذرات پاس نشد. کمترین فاصله: ${current.minimum_pair_distance.toFixed(5)}`:`در بودجهٔ جستجوی فعلی جواب مطابق معیارها پیدا نشد. ${current.reason}`;
$('meta').textContent=`Re=${DATA.settings.re}\nd_p=${DATA.settings.diameter}\nrho_p/rho_f=${DATA.settings.density}\nposition tolerance=${DATA.settings.target_tol}\nlookup relative L2=${DATA.lookup_error.toExponential(3)}\nFinal paths: direct PINN + assumed extension`;refresh();}
$('shape').onchange=choose;$('play').onclick=()=>{if(current?.target_time&&clock>=current.target_time)clock=0;playing=!playing;last=null;};$('again').onclick=()=>{clock=0;playing=true;last=null;refresh();};$('time').oninput=()=>{clock=Number($('time').value)*(current.target_time||1);playing=false;refresh();};
$('csv').onclick=()=>{let rows=current.schedules||[];if(!rows.length)return;let keys=Object.keys(rows[0]),text=keys.join(',')+'\n'+rows.map(r=>keys.map(k=>r[k]).join(',')).join('\n'),url=URL.createObjectURL(new Blob([text],{type:'text/csv'})),a=document.createElement('a');a.href=url;a.download='release_schedule_'+current.kind+'.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};
function tick(now){let T=current?.target_time||0;if(last!==null&&playing&&T>0){clock=Math.min(T,clock+Math.min(.1,(now-last)/1000)*Number($('speed').value));if(clock>=T)playing=false;}last=now;refresh();requestAnimationFrame(tick);}const picker=$('shape');picker.innerHTML='';for(const key of Object.keys(DATA.results)){let option=document.createElement('option');option.value=key;option.textContent=key==='triangle'?'مثلث — ۳ ذره':key==='square'?'مربع — ۴ ذره':key.startsWith('circle_')?'دایره — '+key.split('_')[1]+' ذره':key.startsWith('s_')?'S — '+key.split('_')[1]+' ذره':key;picker.appendChild(option);}picker.value=Object.keys(DATA.results)[0];new ResizeObserver(refresh).observe($('scene'));new ResizeObserver(refresh).observe($('zoom'));choose();requestAnimationFrame(tick);
</script></html>
'''
_fc_page=_FC_HTML.replace('__DATA__',json.dumps(_fc_data,separators=(',',':'),allow_nan=False))
_fc_html_path=_fc_out/'formation_control.html';_fc_html_path.write_text(_fc_page,encoding='utf-8')
display(HTML('<iframe title="Optimized particle release" style="width:100%;height:1150px;border:0" sandbox="allow-scripts allow-downloads" srcdoc="'+html.escape(_fc_page,quote=True)+'"></iframe>'))
_fc_archive=shutil.make_archive(str(_fc_out),'zip',root_dir=_fc_out)
print('Download results and standalone web app:')
display(FileLink(str(_fc_html_path)))
display(FileLink(_fc_archive))
